# Full-code Google Colab Notebook

**Dự báo thời tiết Hà Nội lúc 12:00 bằng Machine Learning**

Notebook này chứa toàn bộ source code cần thiết để chạy lại pipeline và tạo ra các bảng số liệu, biểu đồ, báo cáo tương tự final academic report.

Quy trình chạy:

1. Cài thư viện.
2. Upload file dữ liệu `open_meteo_hanoi.csv` nếu notebook chưa thấy dữ liệu.
3. Ghi toàn bộ source code vào runtime Colab.
4. Chạy pipeline dự báo ngày kế tiếp.
5. Chạy pipeline dự báo 7 ngày với `n_iter=4` và `n_iter=20`.
6. Sinh biểu đồ tuning/model comparison và báo cáo tổng hợp.

Lưu ý: chạy đầy đủ bản `n_iter=20` có thể mất vài phút vì huấn luyện nhiều mô hình XGBoost qua nhiều horizon.


## 1. Cài đặt thư viện


In [ ]:
# Colab thường đã có numpy/pandas/sklearn/matplotlib. Cell này cài thêm các gói cần thiết.
!pip -q install xgboost joblib tabulate nbformat

# Pandoc chỉ cần nếu muốn xuất thêm DOCX từ Markdown.
!apt-get -qq update
!apt-get -qq install -y pandoc


## 2. Chuẩn bị dữ liệu

Nếu bạn upload cả project lên Colab và đã có `data/raw/open_meteo_hanoi.csv`, cell này sẽ dùng luôn file đó. Nếu chưa có, Colab sẽ yêu cầu upload file CSV dữ liệu Open-Meteo.


In [ ]:
from pathlib import Path

Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('reports').mkdir(parents=True, exist_ok=True)
Path('scripts').mkdir(parents=True, exist_ok=True)
Path('src').mkdir(parents=True, exist_ok=True)

data_path = Path('data/raw/open_meteo_hanoi.csv')
if not data_path.exists():
    try:
        from google.colab import files
        print('Upload file dữ liệu CSV Open-Meteo, ví dụ: open_meteo_hanoi.csv')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('Chưa upload file dữ liệu.')
        first_name = next(iter(uploaded.keys()))
        data_path.write_bytes(uploaded[first_name])
        print(f'Saved uploaded file to {data_path}')
    except ModuleNotFoundError:
        raise FileNotFoundError('Không tìm thấy data/raw/open_meteo_hanoi.csv. Hãy đặt file dữ liệu vào đúng đường dẫn này.')
else:
    print(f'Found data file: {data_path}')


## 3. Ghi toàn bộ source code vào notebook runtime

Các cell dưới đây tạo lại cấu trúc source code của project trong môi trường Colab. Đây là phần giúp notebook có thể chạy độc lập mà không cần import từ repo bên ngoài.


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/config.py
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parents[1]
RAW_DATA_PATH = ROOT_DIR / "data" / "raw" / "open_meteo_hanoi.csv"
FALLBACK_RAW_DATA_PATH = ROOT_DIR / "open-meteo-21.05N105.81E18m - open-meteo-21.05N105.81E18m.csv"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
PROCESSED_DATA_PATH = PROCESSED_DIR / "noon_weather_cleaned.csv"
MODELS_DIR = ROOT_DIR / "models"
TEMP_MODEL_DIR = MODELS_DIR / "temperature_regression"
RAIN_MODEL_DIR = MODELS_DIR / "rain_classification"
REPORTS_DIR = ROOT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"
FINAL_REPORT_PATH = REPORTS_DIR / "final_results.md"

RANDOM_STATE = 42
N_SPLITS = 5
SEARCH_ITER = 4

DROPPED_COLUMNS = ["surface_pressure", "wind_direction_10m", "rain"]
TARGET_COLUMNS = [
    "target_temperature",
    "target_precipitation",
    "target_rain",
    "target_weather_level",
]


def ensure_directories() -> None:
    for path in [
        PROCESSED_DIR,
        TEMP_MODEL_DIR,
        RAIN_MODEL_DIR,
        FIGURES_DIR,
        TABLES_DIR,
    ]:
        path.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile src/utils.py
import time
from contextlib import contextmanager
from pathlib import Path

import joblib


@contextmanager
def timer():
    start = time.perf_counter()
    result = {"seconds": 0.0}
    try:
        yield result
    finally:
        result["seconds"] = time.perf_counter() - start


def save_joblib(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(obj, path)


def weather_level(temperature: float) -> str:
    if temperature < 22:
        return "cool"
    if temperature <= 30:
        return "normal"
    return "hot"


In [ ]:
%%writefile src/data_loader.py
from pathlib import Path

import pandas as pd


class DataLoader:
    def __init__(self, csv_path: Path):
        self.csv_path = Path(csv_path)

    def load_raw_data(self) -> pd.DataFrame:
        header_row = self._find_header_row()
        return pd.read_csv(self.csv_path, skiprows=header_row)

    def normalize_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        rename_map = {
            "temperature_2m (degC)": "temperature",
            "temperature_2m (°C)": "temperature",
            "relative_humidity_2m (%)": "humidity",
            "precipitation (mm)": "precipitation",
            "rain (mm)": "rain",
            "cloud_cover (%)": "cloud_cover",
            "pressure_msl (hPa)": "pressure_msl",
            "surface_pressure (hPa)": "surface_pressure",
            "wind_speed_10m (km/h)": "wind_speed_10m",
            "wind_direction_10m (°)": "wind_direction_10m",
            "shortwave_radiation (W/m²)": "shortwave_radiation",
        }
        normalized = []
        for col in df.columns:
            clean = str(col).strip()
            normalized.append(rename_map.get(clean, clean))
        df = df.copy()
        df.columns = normalized
        return df

    def parse_datetime(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        df = df.dropna(subset=["time"])
        df = df.sort_values("time")
        df = df.drop_duplicates(subset=["time"], keep="first")
        return df.reset_index(drop=True)

    def load(self) -> pd.DataFrame:
        df = self.load_raw_data()
        df = self.normalize_columns(df)
        return self.parse_datetime(df)

    def _find_header_row(self) -> int:
        with self.csv_path.open("r", encoding="utf-8") as handle:
            for i, line in enumerate(handle):
                if line.lower().startswith("time,"):
                    return i
        raise ValueError(f"Could not find Open-Meteo table header in {self.csv_path}")


In [ ]:
%%writefile src/preprocessing.py
import pandas as pd

from .config import DROPPED_COLUMNS, PROCESSED_DATA_PATH


class Preprocessor:
    def filter_noon_records(self, df: pd.DataFrame) -> pd.DataFrame:
        noon = df[df["time"].dt.hour == 12].copy()
        return noon.reset_index(drop=True)

    def preprocess(self, df: pd.DataFrame, save: bool = True) -> pd.DataFrame:
        df = self.filter_noon_records(df)
        df = df.drop(columns=[c for c in DROPPED_COLUMNS if c in df.columns])
        numeric_cols = df.select_dtypes(include="number").columns
        for col in numeric_cols:
            df[col] = df[col].fillna(df[col].median())
        if save:
            PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(PROCESSED_DATA_PATH, index=False)
        return df


In [ ]:
%%writefile src/feature_engineering.py
import numpy as np
import pandas as pd

from .utils import weather_level


class FeatureEngineer:
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy().sort_values("time").reset_index(drop=True)
        self._add_time_features(df)
        self._add_lag_features(df)
        self._add_rolling_features(df)
        self._add_targets(df)
        df = df.dropna(subset=["target_temperature", "target_precipitation", "target_rain"])
        numeric_cols = df.select_dtypes(include="number").columns
        for col in numeric_cols:
            df[col] = df[col].fillna(df[col].median())
        return df.reset_index(drop=True)

    def _add_time_features(self, df: pd.DataFrame) -> None:
        df["month"] = df["time"].dt.month
        df["day_of_year"] = df["time"].dt.dayofyear
        df["season"] = np.select(
            [
                df["month"].isin([12, 1, 2]),
                df["month"].isin([3, 4, 5]),
                df["month"].isin([6, 7, 8]),
                df["month"].isin([9, 10, 11]),
            ],
            [0, 1, 2, 3],
            default=0,
        )

    def _add_lag_features(self, df: pd.DataFrame) -> None:
        for col, prefix in [
            ("temperature", "temp"),
            ("humidity", "humidity"),
            ("precipitation", "rain"),
        ]:
            if col in df.columns:
                for lag in [1, 3, 7]:
                    df[f"{prefix}_lag{lag}"] = df[col].shift(lag)

    def _add_rolling_features(self, df: pd.DataFrame) -> None:
        df["temp_roll_mean_3"] = df["temperature"].shift(1).rolling(3).mean()
        df["temp_roll_mean_7"] = df["temperature"].shift(1).rolling(7).mean()
        df["humidity_roll_mean_3"] = df["humidity"].shift(1).rolling(3).mean()
        df["humidity_roll_mean_7"] = df["humidity"].shift(1).rolling(7).mean()
        df["rain_roll_sum_3"] = df["precipitation"].shift(1).rolling(3).sum()
        df["rain_roll_sum_7"] = df["precipitation"].shift(1).rolling(7).sum()

    def _add_targets(self, df: pd.DataFrame) -> None:
        df["target_temperature"] = df["temperature"].shift(-1)
        df["target_precipitation"] = df["precipitation"].shift(-1)
        df["target_rain"] = (df["target_precipitation"] > 0).astype(float)
        df["target_weather_level"] = df["target_temperature"].apply(
            lambda value: weather_level(value) if pd.notna(value) else np.nan
        )


In [ ]:
%%writefile src/split.py
import pandas as pd


def chronological_split(df: pd.DataFrame):
    df = df.sort_values("time").reset_index(drop=True)
    years = set(df["time"].dt.year)
    if {2020, 2021, 2022, 2023, 2024, 2025}.issubset(years):
        train = df[df["time"].dt.year <= 2023]
        val = df[df["time"].dt.year == 2024]
        test = df[df["time"].dt.year == 2025]
        return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    return (
        df.iloc[:train_end].reset_index(drop=True),
        df.iloc[train_end:val_end].reset_index(drop=True),
        df.iloc[val_end:].reset_index(drop=True),
    )


In [ ]:
%%writefile src/baseline.py
import time

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)


def regression_baseline(test_df: pd.DataFrame) -> dict:
    start = time.perf_counter()
    y_true = test_df["target_temperature"]
    y_pred = test_df["temperature"]
    return {
        "model": "Baseline_TodayTemperature",
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": r2_score(y_true, y_pred),
        "training_time": time.perf_counter() - start,
    }


def rain_baseline(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict:
    start = time.perf_counter()
    majority = int(train_df["target_rain"].mode().iloc[0])
    y_true = test_df["target_rain"].astype(int)
    y_pred = np.full(len(test_df), majority)
    return {
        "model": "Baseline_MajorityClass",
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_pred) if len(set(y_true)) == 2 else np.nan,
        "training_time": time.perf_counter() - start,
    }


In [ ]:
%%writefile src/train_temperature.py
import time

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import N_SPLITS, RANDOM_STATE, SEARCH_ITER

try:
    from xgboost import XGBRegressor
except ImportError:  # pragma: no cover - used only when dependency is unavailable.
    XGBRegressor = None


def train_temperature_models(X_train, y_train, X_test, y_test) -> tuple[pd.DataFrame, dict]:
    models = {
        "LinearRegression": Pipeline(
            [("scaler", StandardScaler()), ("model", LinearRegression())]
        ),
        "KNNRegressor": Pipeline(
            [("scaler", StandardScaler()), ("model", KNeighborsRegressor())]
        ),
    }
    searches = {
        "KNNRegressor": {
            "model__n_neighbors": randint(3, 26),
            "model__weights": ["uniform", "distance"],
        }
    }
    if XGBRegressor is not None:
        models["XGBoostRegressor"] = XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
        )
        searches["XGBoostRegressor"] = {
            "n_estimators": randint(100, 501),
            "max_depth": randint(3, 11),
            "learning_rate": uniform(0.01, 0.29),
        }

    cv = TimeSeriesSplit(n_splits=min(N_SPLITS, max(2, len(X_train) // 120)))
    rows = []
    fitted = {}
    for name, estimator in models.items():
        start = time.perf_counter()
        if name in searches:
            estimator = RandomizedSearchCV(
                estimator,
                searches[name],
                n_iter=SEARCH_ITER,
                cv=cv,
                scoring="neg_root_mean_squared_error",
                random_state=RANDOM_STATE,
                n_jobs=1,
            )
        estimator.fit(X_train, y_train)
        training_time = time.perf_counter() - start
        y_pred = estimator.predict(X_test)
        rows.append(
            {
                "model": name,
                "mae": mean_absolute_error(y_test, y_pred),
                "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
                "r2": r2_score(y_test, y_pred),
                "training_time": training_time,
            }
        )
        fitted[name] = {"model": estimator, "predictions": y_pred}
    return pd.DataFrame(rows), fitted


In [ ]:
%%writefile src/train_rain.py
import time

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform, loguniform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from .config import N_SPLITS, RANDOM_STATE, SEARCH_ITER

try:
    from xgboost import XGBClassifier
except ImportError:  # pragma: no cover - used only when dependency is unavailable.
    XGBClassifier = None


def train_rain_models(X_train, y_train, X_test, y_test) -> tuple[pd.DataFrame, dict]:
    y_train = y_train.astype(int)
    y_test = y_test.astype(int)
    models = {
        "LogisticRegression": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "SVM": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "model",
                    SVC(
                        class_weight="balanced",
                        probability=True,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }
    searches = {
        "LogisticRegression": {"model__C": loguniform(0.01, 100)},
        "SVM": {"model__C": loguniform(0.1, 100), "model__kernel": ["linear", "rbf"]},
    }
    if XGBClassifier is not None:
        neg = max(1, int((y_train == 0).sum()))
        pos = max(1, int((y_train == 1).sum()))
        models["XGBoostClassifier"] = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=neg / pos,
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
        )
        searches["XGBoostClassifier"] = {
            "n_estimators": randint(100, 501),
            "max_depth": randint(3, 11),
            "learning_rate": uniform(0.01, 0.29),
        }

    cv = TimeSeriesSplit(n_splits=min(N_SPLITS, max(2, len(X_train) // 120)))
    rows = []
    fitted = {}
    for name, estimator in models.items():
        start = time.perf_counter()
        estimator = RandomizedSearchCV(
            estimator,
            searches[name],
            n_iter=SEARCH_ITER,
            cv=cv,
            scoring="f1",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
        estimator.fit(X_train, y_train)
        training_time = time.perf_counter() - start
        y_pred = estimator.predict(X_test)
        y_score = _prediction_scores(estimator, X_test, y_pred)
        rows.append(
            {
                "model": name,
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred, zero_division=0),
                "recall": recall_score(y_test, y_pred, zero_division=0),
                "f1": f1_score(y_test, y_pred, zero_division=0),
                "roc_auc": roc_auc_score(y_test, y_score)
                if len(set(y_test)) == 2
                else np.nan,
                "training_time": training_time,
            }
        )
        fitted[name] = {"model": estimator, "predictions": y_pred, "scores": y_score}
    return pd.DataFrame(rows), fitted


def _prediction_scores(estimator, X, fallback):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X)[:, 1]
    if hasattr(estimator, "decision_function"):
        return estimator.decision_function(X)
    return fallback


In [ ]:
%%writefile src/evaluate.py
import pandas as pd

from .config import RAIN_MODEL_DIR, TABLES_DIR, TEMP_MODEL_DIR
from .utils import save_joblib


def save_results(
    validation_regression_results: pd.DataFrame,
    validation_rain_results: pd.DataFrame,
    test_regression_results: pd.DataFrame,
    test_rain_results: pd.DataFrame,
    baseline_results: pd.DataFrame,
    experiment_results: pd.DataFrame,
    fitted_temperature: dict,
    fitted_rain: dict,
) -> tuple[str, str]:
    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    validation_regression_results.to_csv(TABLES_DIR / "validation_regression_results.csv", index=False)
    validation_rain_results.to_csv(TABLES_DIR / "validation_rain_classification_results.csv", index=False)
    test_regression_results.to_csv(TABLES_DIR / "regression_results.csv", index=False)
    test_rain_results.to_csv(TABLES_DIR / "rain_classification_results.csv", index=False)
    validation_combined = pd.concat(
        [
            validation_regression_results.assign(task="temperature_regression"),
            validation_rain_results.assign(task="rain_classification"),
        ],
        ignore_index=True,
        sort=False,
    )
    test_combined = pd.concat(
        [
            test_regression_results.assign(task="temperature_regression"),
            test_rain_results.assign(task="rain_classification"),
        ],
        ignore_index=True,
        sort=False,
    )
    validation_combined.to_csv(TABLES_DIR / "validation_results.csv", index=False)
    test_combined.to_csv(TABLES_DIR / "test_results.csv", index=False)
    baseline_results.to_csv(TABLES_DIR / "baseline_results.csv", index=False)
    experiment_results.to_csv(TABLES_DIR / "experiment_ab_comparison.csv", index=False)
    _baseline_vs_ml(test_regression_results, test_rain_results, baseline_results).to_csv(
        TABLES_DIR / "baseline_vs_ml_comparison.csv", index=False
    )
    _runtime(test_regression_results, test_rain_results, baseline_results).to_csv(
        TABLES_DIR / "runtime_comparison.csv", index=False
    )

    best_temp = validation_regression_results.sort_values("rmse").iloc[0]["model"]
    best_rain = validation_rain_results.sort_values("f1", ascending=False).iloc[0]["model"]
    save_joblib(fitted_temperature[best_temp]["model"], TEMP_MODEL_DIR / "best_temperature_model.joblib")
    save_joblib(fitted_rain[best_rain]["model"], RAIN_MODEL_DIR / "best_rain_model.joblib")
    return best_temp, best_rain


def _baseline_vs_ml(regression_results, rain_results, baseline_results):
    rows = []
    for _, row in baseline_results.iterrows():
        rows.append({"task": row["task"], "model": row["model"], "primary_metric": row["primary_metric"], "value": row["value"]})
    rows.append(
        {
            "task": "temperature_regression",
            "model": regression_results.sort_values("rmse").iloc[0]["model"],
            "primary_metric": "rmse",
            "value": regression_results["rmse"].min(),
        }
    )
    rows.append(
        {
            "task": "rain_classification",
            "model": rain_results.sort_values("f1", ascending=False).iloc[0]["model"],
            "primary_metric": "f1",
            "value": rain_results["f1"].max(),
        }
    )
    return pd.DataFrame(rows)


def _runtime(regression_results, rain_results, baseline_results):
    baseline = baseline_results[["task", "model", "training_time"]]
    regression = regression_results[["model", "training_time"]].assign(task="temperature_regression")
    rain = rain_results[["model", "training_time"]].assign(task="rain_classification")
    return pd.concat([baseline, regression, rain], ignore_index=True)


In [ ]:
%%writefile src/visualization.py
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.preprocessing import StandardScaler

from .config import FIGURES_DIR, TABLES_DIR

sns.set_theme(style="whitegrid")


def visualize_eda(df: pd.DataFrame, feature_df: pd.DataFrame, feature_cols: list[str]) -> None:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    _line(df, "time", "temperature", "Noon temperature trend", "temperature_trend.png")
    _hist(df, "temperature", "Noon temperature distribution", "temperature_histogram.png")

    monthly = df.assign(month=df["time"].dt.month)
    _bar(
        monthly.groupby("month", as_index=False)["temperature"].mean(),
        "month",
        "temperature",
        "Monthly average noon temperature",
        "monthly_temperature.png",
    )
    _bar(
        monthly.groupby("month", as_index=False)["precipitation"].sum(),
        "month",
        "precipitation",
        "Monthly rainfall at noon",
        "monthly_rainfall.png",
    )

    rain_dist = feature_df.assign(rain_label=feature_df["target_rain"].map({0.0: "No rain", 1.0: "Rain"}))
    _count(rain_dist, "rain_label", "Rain class distribution", "rain_distribution.png")

    plt.figure(figsize=(12, 9))
    corr = feature_df[feature_cols].corr(numeric_only=True)
    sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.2)
    plt.title("Numeric feature correlation heatmap")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=160)
    plt.close()

    X_scaled = StandardScaler().fit_transform(feature_df[feature_cols])
    pca = PCA(n_components=2, random_state=42)
    components = pca.fit_transform(X_scaled)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=feature_df["target_rain"], palette="Set1", s=24)
    plt.title("PCA visualization of engineered weather features")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "pca_visualization.png", dpi=160)
    plt.close()


def visualize_results(
    regression_results: pd.DataFrame,
    rain_results: pd.DataFrame,
    experiment_results: pd.DataFrame,
    y_temp_true,
    y_temp_pred,
    y_rain_true,
    y_rain_pred,
    feature_importance: pd.DataFrame,
) -> None:
    _metric_bar(regression_results, ["mae", "rmse", "r2"], "Regression model comparison", "regression_model_comparison.png")
    _metric_bar(
        rain_results,
        ["accuracy", "precision", "recall", "f1", "roc_auc"],
        "Rain classification model comparison",
        "classification_model_comparison.png",
    )

    plt.figure(figsize=(9, 6))
    plt.scatter(y_temp_true, y_temp_pred, alpha=0.65)
    low = min(np.min(y_temp_true), np.min(y_temp_pred))
    high = max(np.max(y_temp_true), np.max(y_temp_pred))
    plt.plot([low, high], [low, high], color="red", linewidth=1)
    plt.title("Actual vs predicted next-day noon temperature")
    plt.xlabel("Actual temperature")
    plt.ylabel("Predicted temperature")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "actual_vs_predicted_temperature.png", dpi=160)
    plt.close()

    residuals = np.asarray(y_temp_true) - np.asarray(y_temp_pred)
    plt.figure(figsize=(8, 5))
    sns.histplot(residuals, kde=True)
    plt.title("Temperature prediction residuals")
    plt.xlabel("Residual")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "temperature_residuals.png", dpi=160)
    plt.close()

    cm = confusion_matrix(y_rain_true, y_rain_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No rain", "Rain"]).plot(cmap="Blues")
    plt.title("Rain classification confusion matrix")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "confusion_matrix_rain.png", dpi=160)
    plt.close()

    if not feature_importance.empty:
        top = feature_importance.head(10)
        plt.figure(figsize=(8, 6))
        sns.barplot(data=top, x="importance", y="feature")
        plt.title("Top XGBoost feature importance")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "feature_importance.png", dpi=160)
        plt.close()
    else:
        plt.figure(figsize=(8, 4))
        plt.text(0.5, 0.5, "Feature importance unavailable", ha="center", va="center")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "feature_importance.png", dpi=160)
        plt.close()

    _experiment_plot(experiment_results)


def visualize_validation_test_predictions(
    temperature_validation: pd.DataFrame,
    temperature_test: pd.DataFrame,
    rain_validation: pd.DataFrame,
    rain_test: pd.DataFrame,
) -> None:
    _actual_vs_predicted_split(
        temperature_validation,
        "Validation actual vs predicted next-day noon temperature",
        "validation_actual_vs_predicted_temperature.png",
    )
    _actual_vs_predicted_split(
        temperature_test,
        "Test actual vs predicted next-day noon temperature",
        "test_actual_vs_predicted_temperature.png",
    )
    _residual_split(
        temperature_validation,
        "Validation temperature residuals",
        "validation_temperature_residuals.png",
    )
    _residual_split(
        temperature_test,
        "Test temperature residuals",
        "test_temperature_residuals.png",
    )
    _temperature_tolerance_plot(temperature_validation, temperature_test)
    _rain_confusion_split(
        rain_validation,
        "Validation rain confusion matrix",
        "validation_confusion_matrix_rain.png",
    )
    _rain_confusion_split(
        rain_test,
        "Test rain confusion matrix",
        "test_confusion_matrix_rain.png",
    )
    _rain_correct_wrong_plot(rain_validation, rain_test)


def visualize_temperature_model_predictions(
    fitted_temperature: dict,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> None:
    val_predictions = {}
    test_predictions = {}
    val_metrics = {}
    test_metrics = {}
    for model_name, payload in fitted_temperature.items():
        safe_name = _safe_model_name(model_name)
        model = payload["model"]
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)
        val_predictions[model_name] = val_pred
        test_predictions[model_name] = test_pred
        val_metrics[model_name] = _regression_metric_summary(y_val, val_pred)
        test_metrics[model_name] = _regression_metric_summary(y_test, test_pred)
        _actual_vs_predicted_arrays(
            y_val,
            val_pred,
            f"Validation actual vs predicted temperature - {model_name}\n{_metric_text(val_metrics[model_name])}",
            f"validation_actual_vs_predicted_{safe_name}.png",
        )
        _actual_vs_predicted_arrays(
            y_test,
            test_pred,
            f"Test actual vs predicted temperature - {model_name}\n{_metric_text(test_metrics[model_name])}",
            f"test_actual_vs_predicted_{safe_name}.png",
        )

    _actual_vs_predicted_grid(
        y_val,
        val_predictions,
        val_metrics,
        "Validation actual vs predicted temperature by model",
        "validation_actual_vs_predicted_all_regression_models.png",
    )
    _actual_vs_predicted_grid(
        y_test,
        test_predictions,
        test_metrics,
        "Test actual vs predicted temperature by model",
        "test_actual_vs_predicted_all_regression_models.png",
    )
    _absolute_error_boxplot(
        y_val,
        val_predictions,
        "Validation absolute error distribution by regression model",
        "validation_temperature_absolute_error_by_model.png",
    )
    _absolute_error_boxplot(
        y_test,
        test_predictions,
        "Test absolute error distribution by regression model",
        "test_temperature_absolute_error_by_model.png",
    )


def save_feature_importance(model, feature_cols: list[str]) -> pd.DataFrame:
    estimator = getattr(model, "best_estimator_", model)
    if hasattr(estimator, "named_steps"):
        estimator = estimator.named_steps.get("model", estimator)
    importances = getattr(estimator, "feature_importances_", None)
    if importances is None:
        result = pd.DataFrame(columns=["feature", "importance"])
    else:
        result = (
            pd.DataFrame({"feature": feature_cols, "importance": importances})
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )
    result.head(10).to_csv(TABLES_DIR / "top_10_features.csv", index=False)
    return result


def _line(df, x, y, title, filename):
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=df, x=x, y=y, linewidth=1)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _hist(df, col, title, filename):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col], kde=True)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _bar(df, x, y, title, filename):
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df, x=x, y=y)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _count(df, x, title, filename):
    plt.figure(figsize=(6, 5))
    sns.countplot(data=df, x=x)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _metric_bar(results, metrics, title, filename):
    melted = results.melt(id_vars="model", value_vars=metrics, var_name="metric", value_name="value")
    plt.figure(figsize=(10, 5))
    sns.barplot(data=melted, x="model", y="value", hue="metric")
    plt.title(title)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _experiment_plot(experiment_results):
    available = [
        c
        for c in [
            "best_validation_regression_rmse",
            "test_rmse_for_validation_best",
            "best_validation_rain_f1",
            "test_f1_for_validation_best",
        ]
        if c in experiment_results.columns
    ]
    melted = experiment_results.melt(id_vars="experiment", value_vars=available, var_name="metric", value_name="value")
    plt.figure(figsize=(8, 5))
    sns.barplot(data=melted, x="experiment", y="value", hue="metric")
    plt.title("Experiment A/B comparison")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "experiment_ab_comparison.png", dpi=160)
    plt.close()


def _actual_vs_predicted_split(df, title, filename):
    plt.figure(figsize=(9, 6))
    plt.scatter(df["actual_temperature"], df["predicted_temperature"], alpha=0.65)
    low = min(df["actual_temperature"].min(), df["predicted_temperature"].min())
    high = max(df["actual_temperature"].max(), df["predicted_temperature"].max())
    plt.plot([low, high], [low, high], color="red", linewidth=1)
    plt.title(title)
    plt.xlabel("Actual temperature")
    plt.ylabel("Predicted temperature")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _actual_vs_predicted_arrays(y_true, y_pred, title, filename):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.65)
    low = min(y_true.min(), y_pred.min())
    high = max(y_true.max(), y_pred.max())
    plt.plot([low, high], [low, high], color="red", linewidth=1)
    plt.title(title)
    plt.xlabel("Actual temperature")
    plt.ylabel("Predicted temperature")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _actual_vs_predicted_grid(y_true, predictions: dict, metrics: dict, title: str, filename: str):
    y_true = np.asarray(y_true)
    n_models = len(predictions)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), sharex=True, sharey=True)
    if n_models == 1:
        axes = [axes]
    all_pred = np.concatenate([np.asarray(pred) for pred in predictions.values()])
    low = min(y_true.min(), all_pred.min())
    high = max(y_true.max(), all_pred.max())
    for ax, (model_name, pred) in zip(axes, predictions.items()):
        pred = np.asarray(pred)
        ax.scatter(y_true, pred, alpha=0.65)
        ax.plot([low, high], [low, high], color="red", linewidth=1)
        ax.set_title(f"{model_name}\n{_metric_text(metrics[model_name])}")
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _safe_model_name(model_name: str) -> str:
    return (
        model_name.replace("LinearRegression", "linear_regression")
        .replace("KNNRegressor", "knn_regressor")
        .replace("XGBoostRegressor", "xgboost_regressor")
        .lower()
    )


def _regression_metric_summary(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    residual = y_true - y_pred
    mae = np.mean(np.abs(residual))
    rmse = np.sqrt(np.mean(residual ** 2))
    ss_res = np.sum(residual ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot else np.nan
    return {"mae": mae, "rmse": rmse, "r2": r2}


def _metric_text(metrics):
    return f"MAE={metrics['mae']:.3f}, RMSE={metrics['rmse']:.3f}, R2={metrics['r2']:.3f}"


def _absolute_error_boxplot(y_true, predictions: dict, title: str, filename: str):
    y_true = np.asarray(y_true)
    rows = []
    for model_name, pred in predictions.items():
        pred = np.asarray(pred)
        for error in np.abs(y_true - pred):
            rows.append({"model": model_name, "absolute_error": error})
    df = pd.DataFrame(rows)
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x="model", y="absolute_error")
    sns.stripplot(data=df, x="model", y="absolute_error", color="black", alpha=0.18, size=2)
    plt.title(title)
    plt.xlabel("Model")
    plt.ylabel("Absolute error (°C)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _residual_split(df, title, filename):
    plt.figure(figsize=(8, 5))
    sns.histplot(df["residual"], kde=True)
    plt.axvline(0, color="red", linewidth=1)
    plt.title(title)
    plt.xlabel("Residual")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _temperature_tolerance_plot(validation_df, test_df):
    rows = []
    for split, df in [("validation", validation_df), ("test", test_df)]:
        for tolerance in [1, 2, 3]:
            rows.append(
                {
                    "split": split,
                    "tolerance": f"within ±{tolerance}°C",
                    "rate": df[f"within_{tolerance}c"].mean(),
                }
            )
    plt.figure(figsize=(8, 5))
    sns.barplot(data=pd.DataFrame(rows), x="tolerance", y="rate", hue="split")
    plt.ylim(0, 1)
    plt.title("Temperature prediction tolerance accuracy")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "temperature_tolerance_accuracy.png", dpi=160)
    plt.close()


def _rain_confusion_split(df, title, filename):
    cm = confusion_matrix(df["actual_rain"], df["predicted_rain"])
    ConfusionMatrixDisplay(cm, display_labels=["No rain", "Rain"]).plot(cmap="Blues")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.close()


def _rain_correct_wrong_plot(validation_df, test_df):
    rows = []
    for split, df in [("validation", validation_df), ("test", test_df)]:
        rows.append({"split": split, "result": "correct", "rate": df["correct"].mean()})
        rows.append({"split": split, "result": "wrong", "rate": 1 - df["correct"].mean()})
    plt.figure(figsize=(7, 5))
    sns.barplot(data=pd.DataFrame(rows), x="split", y="rate", hue="result")
    plt.ylim(0, 1)
    plt.title("Rain prediction correct vs wrong rate")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "rain_correct_wrong_rate.png", dpi=160)
    plt.close()


In [ ]:
%%writefile src/explainability.py
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.inspection import permutation_importance

from .config import FIGURES_DIR, RANDOM_STATE, TABLES_DIR


def generate_model_explanations(
    fitted_temperature: dict,
    fitted_rain: dict,
    X_test: pd.DataFrame,
    y_temp_test: pd.Series,
    y_rain_test: pd.Series,
    feature_cols: list[str],
) -> dict[str, pd.DataFrame]:
    explanations = {}

    explanations["linear_regression_coefficients"] = coefficient_table(
        fitted_temperature["LinearRegression"]["model"],
        feature_cols,
        task="temperature_regression",
    )
    explanations["linear_regression_coefficients"].to_csv(
        TABLES_DIR / "linear_regression_coefficients.csv", index=False
    )

    explanations["knn_permutation_importance"] = permutation_table(
        fitted_temperature["KNNRegressor"]["model"],
        X_test,
        y_temp_test,
        feature_cols,
        scoring="neg_root_mean_squared_error",
        task="temperature_regression",
    )
    explanations["knn_permutation_importance"].to_csv(
        TABLES_DIR / "knn_permutation_importance.csv", index=False
    )

    if "XGBoostRegressor" in fitted_temperature:
        explanations["xgboost_regressor_feature_importance"] = tree_importance_table(
            fitted_temperature["XGBoostRegressor"]["model"],
            feature_cols,
            task="temperature_regression",
        )
        explanations["xgboost_regressor_feature_importance"].to_csv(
            TABLES_DIR / "xgboost_regressor_feature_importance.csv", index=False
        )

    explanations["logistic_regression_coefficients"] = coefficient_table(
        fitted_rain["LogisticRegression"]["model"],
        feature_cols,
        task="rain_classification",
    )
    explanations["logistic_regression_coefficients"].to_csv(
        TABLES_DIR / "logistic_regression_coefficients.csv", index=False
    )

    explanations["svm_permutation_importance"] = permutation_table(
        fitted_rain["SVM"]["model"],
        X_test,
        y_rain_test,
        feature_cols,
        scoring="f1",
        task="rain_classification",
    )
    explanations["svm_permutation_importance"].to_csv(
        TABLES_DIR / "svm_permutation_importance.csv", index=False
    )

    if "XGBoostClassifier" in fitted_rain:
        explanations["xgboost_classifier_feature_importance"] = tree_importance_table(
            fitted_rain["XGBoostClassifier"]["model"],
            feature_cols,
            task="rain_classification",
        )
        explanations["xgboost_classifier_feature_importance"].to_csv(
            TABLES_DIR / "xgboost_classifier_feature_importance.csv", index=False
        )

    plot_explanations(explanations)
    return explanations


def coefficient_table(model, feature_cols: list[str], task: str) -> pd.DataFrame:
    estimator = getattr(model, "best_estimator_", model)
    if hasattr(estimator, "named_steps"):
        fitted_model = estimator.named_steps["model"]
    else:
        fitted_model = estimator

    coefs = np.asarray(fitted_model.coef_).reshape(-1)
    return (
        pd.DataFrame(
            {
                "task": task,
                "feature": feature_cols,
                "coefficient": coefs,
                "absolute_coefficient": np.abs(coefs),
                "direction": np.where(coefs >= 0, "positive", "negative"),
            }
        )
        .sort_values("absolute_coefficient", ascending=False)
        .reset_index(drop=True)
    )


def permutation_table(
    model,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    feature_cols: list[str],
    scoring: str,
    task: str,
) -> pd.DataFrame:
    result = permutation_importance(
        model,
        X_test,
        y_test,
        scoring=scoring,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    return (
        pd.DataFrame(
            {
                "task": task,
                "feature": feature_cols,
                "importance_mean": result.importances_mean,
                "importance_std": result.importances_std,
            }
        )
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )


def tree_importance_table(model, feature_cols: list[str], task: str) -> pd.DataFrame:
    estimator = getattr(model, "best_estimator_", model)
    if hasattr(estimator, "named_steps"):
        estimator = estimator.named_steps["model"]
    importances = getattr(estimator, "feature_importances_", None)
    if importances is None:
        return pd.DataFrame(columns=["task", "feature", "importance"])
    return (
        pd.DataFrame({"task": task, "feature": feature_cols, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


def plot_explanations(explanations: dict[str, pd.DataFrame]) -> None:
    plot_specs = [
        ("linear_regression_coefficients", "absolute_coefficient", "linear_regression_coefficients.png"),
        ("knn_permutation_importance", "importance_mean", "knn_permutation_importance.png"),
        ("xgboost_regressor_feature_importance", "importance", "xgboost_regressor_feature_importance.png"),
        ("logistic_regression_coefficients", "absolute_coefficient", "logistic_regression_coefficients.png"),
        ("svm_permutation_importance", "importance_mean", "svm_permutation_importance.png"),
        ("xgboost_classifier_feature_importance", "importance", "xgboost_classifier_feature_importance.png"),
    ]
    for key, value_col, filename in plot_specs:
        table = explanations.get(key)
        if table is None or table.empty:
            continue
        top = table.head(10)
        plt.figure(figsize=(8, 6))
        sns.barplot(data=top, x=value_col, y="feature")
        plt.title(key.replace("_", " ").title())
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / filename, dpi=160)
        plt.close()


In [ ]:
%%writefile src/reporting.py
import pandas as pd

from .config import FINAL_REPORT_PATH, FIGURES_DIR


def generate_report(
    raw_df: pd.DataFrame,
    noon_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    best_temp: str,
    best_rain: str,
) -> None:
    FINAL_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    stats = _to_markdown(noon_df.select_dtypes(include="number").describe().round(3).reset_index())
    missing = _to_markdown(raw_df.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_count"}))
    rain_rate = feature_df["target_rain"].mean()
    figures = _figure_links()
    content = f"""# Hanoi Noon Weather Prediction

## Introduction
This project builds a Machine Learning system to predict Hanoi weather at 12:00 PM on the next day.

## Motivation
Noon weather is useful for daily planning because it captures the most active part of the day for commuting, outdoor work, and school schedules.

## Contribution
The project implements two supervised learning tasks: temperature regression and rain binary classification. It also compares baselines with ML models, evaluates an A/B feature experiment for `shortwave_radiation`, analyzes runtime, visualizes the data with PCA, and reports model errors.

## Dataset Overview
- Raw hourly records: {len(raw_df):,}
- Noon records after preprocessing: {len(noon_df):,}
- Engineered supervised samples: {len(feature_df):,}
- Raw date range: {raw_df['time'].min()} to {raw_df['time'].max()}
- Noon date range: {noon_df['time'].min()} to {noon_df['time'].max()}
- Next-day rain rate: {rain_rate:.3f}

## Input and Expected Output
Input: historical 12:00 weather records plus engineered lag and rolling features.

Regression output: next-day 12:00 temperature.

Classification output: next-day rain or no rain.

Derived output: weather level from predicted temperature.

## Project Process / Methodology
The pipeline loads the Open-Meteo CSV, removes metadata rows, normalizes columns, parses timestamps, keeps only 12:00 records, creates supervised learning targets, splits data chronologically, trains baselines and ML models, evaluates the models, saves best models, and writes reports.

## Data Analysis
Missing value summary:

{missing}

Descriptive statistics:

{stats}

Descriptive statistics interpretation:

- `count` confirms that each noon feature has the same number of valid records after preprocessing.
- `mean` shows the average noon condition: temperature is about 26.9°C, humidity about 67.2%, and precipitation about 0.255 mm.
- `std` measures variability. Temperature varies moderately, while precipitation has high variability relative to its mean because most days have no rain and a few days have heavier rain.
- `min` and `max` show the observed range, including temperatures from 8.9°C to 38.0°C and precipitation up to 13.8 mm.
- `25%`, `50%`, and `75%` are quartiles. The median precipitation is 0.0 mm, meaning at least half of noon records have no rainfall.
- Cloud cover has a high median and 75th percentile close to 100%, indicating many noon observations are cloudy.
- Shortwave radiation has a wide range, which makes it useful to test in Experiment A/B because it may explain daytime temperature differences.

## Data Preprocessing
The preprocessing stage removes duplicate timestamps, sorts observations chronologically, keeps only 12:00 records, removes `surface_pressure`, `wind_direction_10m`, and `rain`, fills numeric missing values with medians, and drops only rows where generated targets are unavailable.

## Data Visualization
The project uses standard EDA plots and PCA for low-dimensional visualization.

{figures}

## Feature Engineering
Features include month, day of year, season, temperature/humidity/precipitation lags for 1, 3, and 7 days, and rolling means/sums using `shift(1)` before `rolling()` to avoid target leakage.

## Experimental Protocol
The experiments use a chronological train/validation/test split: 2020-2023 for training, 2024 for validation, and 2025 for final testing. Models are fitted only on the training set. The validation set is used to select the best model and compare experiments. The test set is used only for final evaluation.

Hyperparameter tuning uses `TimeSeriesSplit(n_splits=5)` and `RandomizedSearchCV` on the training data.

## Model Parameter Setup
Temperature models: Linear Regression, KNN Regressor, and XGBoost Regressor. KNN searches neighbors from 3 to 25 and uniform/distance weighting. XGBoost searches estimators, depth, and learning rate.

Rain models: Logistic Regression, SVM, and XGBoost Classifier. Logistic Regression and SVM use balanced class weights. XGBoost uses automatic `scale_pos_weight`.

## Linear Regression Training Process
Linear Regression is the main selected model for temperature prediction. The model is trained only on the training period 2020-2023. The validation year 2024 is used for model selection, and the test year 2025 is used for final evaluation.

The training pipeline uses two steps:

1. `StandardScaler`: standardizes numeric features so that each feature has comparable scale.
2. `LinearRegression`: learns a linear relationship between engineered weather features and next-day noon temperature.

The model assumes the target can be approximated as:

```text
target_temperature = b + w1*x1 + w2*x2 + ... + wn*xn
```

where `x1..xn` are weather features such as current temperature, humidity, pressure, season, lag features, and rolling statistics. During training, Linear Regression estimates the intercept `b` and coefficients `w1..wn` by minimizing the sum of squared prediction errors on the training set.

In this project, the target is:

```text
target_temperature = temperature.shift(-1)
```

Therefore, each training sample uses weather information at 12:00 on one day to predict temperature at 12:00 on the following day. Because the model is solved by least squares rather than iterative gradient descent, no epoch-by-epoch cost curve is reported. Instead, training quality is evaluated using MAE, RMSE, and R2 on validation and test data.

The learned coefficients are reported in `reports/tables/linear_regression_coefficients.csv`. Positive coefficients indicate that increasing the feature tends to increase predicted temperature, while negative coefficients indicate the opposite direction after standardization.

## Baseline Results
See `reports/tables/baseline_results.csv`.

## Regression Results
Best temperature model selected from validation: `{best_temp}`.

- Validation results: `reports/tables/validation_regression_results.csv`
- Final test results: `reports/tables/regression_results.csv`

## Classification Results
Best rain model selected from validation: `{best_rain}`.

- Validation results: `reports/tables/validation_rain_classification_results.csv`
- Final test results: `reports/tables/rain_classification_results.csv`

## Validation and Test Prediction Analysis
This section compares model predictions with actual observations on both validation and test data.

For temperature regression, exact accuracy is not appropriate because the output is continuous. The project therefore reports MAE, RMSE, and tolerance accuracy: the percentage of predictions within ±1°C, ±2°C, and ±3°C of the real temperature.

For rain classification, correctness is direct because the output is binary. The project reports correct/wrong counts, accuracy, precision, recall, F1, and confusion matrices.

Prediction artifacts:

- `reports/tables/temperature_validation_predictions.csv`
- `reports/tables/temperature_test_predictions.csv`
- `reports/tables/rain_validation_predictions.csv`
- `reports/tables/rain_test_predictions.csv`
- `reports/tables/validation_test_prediction_summary.csv`

The validation figures show how the selected model behaves during model selection. The test figures show final performance on unseen data from 2025.

## Runtime / Complexity Analysis
See `reports/tables/runtime_comparison.csv`. Linear models are lightweight, KNN shifts cost toward prediction because it compares neighbors, SVM can be more expensive with nonlinear kernels, and XGBoost spends more time training boosted trees but can capture nonlinear interactions.

## Model Explainability / Feature Importance
Model selection is based on test metrics, while explainability is used to interpret how each trained model uses the input features.

XGBoost feature importance is reported as required by the project specification in `reports/tables/top_10_features.csv` and `reports/figures/feature_importance.png`.

Additional model-specific explanations:

- Linear Regression: `reports/tables/linear_regression_coefficients.csv`
- KNN Regressor: `reports/tables/knn_permutation_importance.csv`
- XGBoost Regressor: `reports/tables/xgboost_regressor_feature_importance.csv`
- Logistic Regression: `reports/tables/logistic_regression_coefficients.csv`
- SVM: `reports/tables/svm_permutation_importance.csv`
- XGBoost Classifier: `reports/tables/xgboost_classifier_feature_importance.csv`

Linear and Logistic Regression use signed coefficients after scaling, where positive values increase the prediction direction and negative values reduce it. KNN and SVM use permutation importance because they do not provide stable built-in feature importance for this setup. XGBoost uses built-in tree-based feature importance.

## Experiment A vs B
See `reports/tables/experiment_ab_comparison.csv` and `reports/figures/experiment_ab_comparison.png` for the comparison with and without `shortwave_radiation`.

## Evaluation and Discussion
The report compares baseline and ML performance in `reports/tables/baseline_vs_ml_comparison.csv`. Regression quality is selected by RMSE, while rain classification is selected by F1 to account for class imbalance.

## Error Analysis for Worst Prediction Days
See `reports/tables/worst_prediction_days.csv` for the highest absolute temperature prediction errors.

## Conclusion
The completed system builds a reproducible ML pipeline for next-day noon weather prediction in Hanoi, including preprocessing, feature engineering, model tuning, visualization, model selection, and reporting.

## Future Work / Perspectives
Future work can add more locations, additional meteorological forecast variables, probability calibration for rain, time-series/deep learning models, and a dashboard or inference API.
"""
    FINAL_REPORT_PATH.write_text(content, encoding="utf-8")


def _figure_links() -> str:
    names = [
        "temperature_trend.png",
        "temperature_histogram.png",
        "monthly_temperature.png",
        "monthly_rainfall.png",
        "rain_distribution.png",
        "correlation_heatmap.png",
        "pca_visualization.png",
        "regression_model_comparison.png",
        "classification_model_comparison.png",
        "actual_vs_predicted_temperature.png",
        "temperature_residuals.png",
        "confusion_matrix_rain.png",
        "validation_actual_vs_predicted_temperature.png",
        "test_actual_vs_predicted_temperature.png",
        "validation_temperature_residuals.png",
        "test_temperature_residuals.png",
        "temperature_tolerance_accuracy.png",
        "validation_confusion_matrix_rain.png",
        "test_confusion_matrix_rain.png",
        "rain_correct_wrong_rate.png",
        "feature_importance.png",
        "linear_regression_coefficients.png",
        "knn_permutation_importance.png",
        "xgboost_regressor_feature_importance.png",
        "logistic_regression_coefficients.png",
        "svm_permutation_importance.png",
        "xgboost_classifier_feature_importance.png",
        "experiment_ab_comparison.png",
    ]
    return "\n".join([f"- ![{name.removesuffix('.png')}](figures/{name})" for name in names])


def _to_markdown(df: pd.DataFrame) -> str:
    columns = [str(c) for c in df.columns]
    rows = ["| " + " | ".join(columns) + " |"]
    rows.append("| " + " | ".join(["---"] * len(columns)) + " |")
    for _, row in df.iterrows():
        rows.append("| " + " | ".join(str(row[c]) for c in df.columns) + " |")
    return "\n".join(rows)


In [ ]:
%%writefile main.py
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)

from src.baseline import rain_baseline, regression_baseline
from src.config import (
    FALLBACK_RAW_DATA_PATH,
    RAW_DATA_PATH,
    TABLES_DIR,
    TARGET_COLUMNS,
    ensure_directories,
)
from src.data_loader import DataLoader
from src.evaluate import save_results
from src.explainability import generate_model_explanations
from src.feature_engineering import FeatureEngineer
from src.preprocessing import Preprocessor
from src.reporting import generate_report
from src.split import chronological_split
from src.train_rain import train_rain_models
from src.train_temperature import train_temperature_models
from src.visualization import (
    save_feature_importance,
    visualize_eda,
    visualize_results,
    visualize_validation_test_predictions,
)


def main() -> None:
    ensure_directories()
    csv_path = RAW_DATA_PATH if RAW_DATA_PATH.exists() else FALLBACK_RAW_DATA_PATH

    raw_df = DataLoader(csv_path).load()
    noon_df = Preprocessor().preprocess(raw_df)
    feature_df = FeatureEngineer().transform(noon_df)
    train_df, val_df, test_df = chronological_split(feature_df)

    experiment_outputs = []
    primary = None
    for include_shortwave, label in [(False, "A_without_shortwave_radiation"), (True, "B_with_shortwave_radiation")]:
        feature_cols = select_feature_columns(feature_df, include_shortwave)
        X_train = train_df[feature_cols]
        y_temp_train = train_df["target_temperature"]
        y_rain_train = train_df["target_rain"]
        X_val = val_df[feature_cols]
        y_temp_val = val_df["target_temperature"]
        y_rain_val = val_df["target_rain"]
        X_test = test_df[feature_cols]
        y_temp_test = test_df["target_temperature"]
        y_rain_test = test_df["target_rain"]

        temp_val_results, fitted_temp = train_temperature_models(X_train, y_temp_train, X_val, y_temp_val)
        rain_val_results, fitted_rain = train_rain_models(X_train, y_rain_train, X_val, y_rain_val)
        temp_test_results = evaluate_temperature_models(fitted_temp, X_test, y_temp_test, temp_val_results)
        rain_test_results = evaluate_rain_models(fitted_rain, X_test, y_rain_test, rain_val_results)
        baseline_results = build_baseline_results(train_df, val_df, test_df)

        experiment_outputs.append(
            {
                "experiment": label,
                "include_shortwave_radiation": include_shortwave,
                "best_regression_model": temp_val_results.sort_values("rmse").iloc[0]["model"],
                "best_validation_regression_rmse": temp_val_results["rmse"].min(),
                "test_rmse_for_validation_best": temp_test_results[
                    temp_test_results["model"] == temp_val_results.sort_values("rmse").iloc[0]["model"]
                ]["rmse"].iloc[0],
                "best_rain_model": rain_val_results.sort_values("f1", ascending=False).iloc[0]["model"],
                "best_validation_rain_f1": rain_val_results["f1"].max(),
                "test_f1_for_validation_best": rain_test_results[
                    rain_test_results["model"] == rain_val_results.sort_values("f1", ascending=False).iloc[0]["model"]
                ]["f1"].iloc[0],
            }
        )

        if include_shortwave:
            primary = {
                "feature_cols": feature_cols,
                "X_val": X_val,
                "X_test": X_test,
                "y_temp_val": y_temp_val,
                "y_temp_test": y_temp_test,
                "y_rain_val": y_rain_val,
                "y_rain_test": y_rain_test,
                "temp_val_results": temp_val_results,
                "rain_val_results": rain_val_results,
                "temp_test_results": temp_test_results,
                "rain_test_results": rain_test_results,
                "baseline_results": baseline_results,
                "fitted_temp": fitted_temp,
                "fitted_rain": fitted_rain,
            }

    experiment_results = pd.DataFrame(experiment_outputs)
    assert primary is not None

    best_temp, best_rain = save_results(
        primary["temp_val_results"],
        primary["rain_val_results"],
        primary["temp_test_results"],
        primary["rain_test_results"],
        primary["baseline_results"],
        experiment_results,
        primary["fitted_temp"],
        primary["fitted_rain"],
    )
    best_temp_val_pred = primary["fitted_temp"][best_temp]["predictions"]
    best_rain_val_pred = primary["fitted_rain"][best_rain]["predictions"]
    best_temp_test_pred = primary["fitted_temp"][best_temp]["model"].predict(primary["X_test"])
    best_rain_test_pred = primary["fitted_rain"][best_rain]["model"].predict(primary["X_test"])
    importance_model_name = (
        "XGBoostRegressor"
        if "XGBoostRegressor" in primary["fitted_temp"]
        else best_temp
    )
    feature_importance = save_feature_importance(
        primary["fitted_temp"][importance_model_name]["model"],
        primary["feature_cols"],
    )
    generate_model_explanations(
        primary["fitted_temp"],
        primary["fitted_rain"],
        primary["X_test"],
        primary["y_temp_test"],
        primary["y_rain_test"],
        primary["feature_cols"],
    )
    prediction_tables = save_prediction_analysis(
        val_df,
        test_df,
        primary["y_temp_val"],
        best_temp_val_pred,
        primary["y_temp_test"],
        best_temp_test_pred,
        primary["y_rain_val"],
        best_rain_val_pred,
        primary["y_rain_test"],
        best_rain_test_pred,
    )
    save_worst_prediction_days(test_df, best_temp_test_pred)
    save_split_summary(train_df, val_df, test_df)

    visualize_eda(noon_df, feature_df, primary["feature_cols"])
    visualize_results(
        primary["temp_test_results"],
        primary["rain_test_results"],
        experiment_results,
        primary["y_temp_test"],
        best_temp_test_pred,
        primary["y_rain_test"],
        best_rain_test_pred,
        feature_importance,
    )
    visualize_validation_test_predictions(
        prediction_tables["temperature_validation"],
        prediction_tables["temperature_test"],
        prediction_tables["rain_validation"],
        prediction_tables["rain_test"],
    )
    generate_report(raw_df, noon_df, feature_df, best_temp, best_rain)
    print("Pipeline completed successfully.")


def select_feature_columns(df: pd.DataFrame, include_shortwave: bool) -> list[str]:
    blocked = set(TARGET_COLUMNS + ["time", "target_precipitation"])
    if not include_shortwave:
        blocked.add("shortwave_radiation")
    numeric = df.select_dtypes(include="number").columns
    return [c for c in numeric if c not in blocked]


def build_baseline_results(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    val_reg = regression_baseline(val_df)
    test_reg = regression_baseline(test_df)
    val_rain = rain_baseline(train_df, val_df)
    test_rain = rain_baseline(train_df, test_df)
    return pd.DataFrame(
        [
            {
                "split": "validation",
                "task": "temperature_regression",
                "model": val_reg["model"],
                "primary_metric": "rmse",
                "value": val_reg["rmse"],
                "training_time": val_reg["training_time"],
                **val_reg,
            },
            {
                "split": "test",
                "task": "temperature_regression",
                "model": test_reg["model"],
                "primary_metric": "rmse",
                "value": test_reg["rmse"],
                "training_time": test_reg["training_time"],
                **test_reg,
            },
            {
                "split": "validation",
                "task": "rain_classification",
                "model": val_rain["model"],
                "primary_metric": "f1",
                "value": val_rain["f1"],
                "training_time": val_rain["training_time"],
                **val_rain,
            },
            {
                "split": "test",
                "task": "rain_classification",
                "model": test_rain["model"],
                "primary_metric": "f1",
                "value": test_rain["f1"],
                "training_time": test_rain["training_time"],
                **test_rain,
            },
        ]
    )


def evaluate_temperature_models(fitted_models, X, y, validation_results):
    rows = []
    training_times = validation_results.set_index("model")["training_time"].to_dict()
    for name, data in fitted_models.items():
        pred = data["model"].predict(X)
        rows.append(
            {
                "model": name,
                "mae": mean_absolute_error(y, pred),
                "rmse": float(np.sqrt(mean_squared_error(y, pred))),
                "r2": r2_score(y, pred),
                "training_time": training_times.get(name, 0.0),
            }
        )
    return pd.DataFrame(rows)


def evaluate_rain_models(fitted_models, X, y, validation_results):
    rows = []
    y = y.astype(int)
    training_times = validation_results.set_index("model")["training_time"].to_dict()
    for name, data in fitted_models.items():
        model = data["model"]
        pred = model.predict(X).astype(int)
        score = prediction_scores(model, X, pred)
        rows.append(
            {
                "model": name,
                "accuracy": accuracy_score(y, pred),
                "precision": precision_score(y, pred, zero_division=0),
                "recall": recall_score(y, pred, zero_division=0),
                "f1": f1_score(y, pred, zero_division=0),
                "roc_auc": roc_auc_score(y, score) if len(set(y)) == 2 else np.nan,
                "training_time": training_times.get(name, 0.0),
            }
        )
    return pd.DataFrame(rows)


def prediction_scores(model, X, fallback):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return fallback


def save_prediction_analysis(
    val_df,
    test_df,
    y_temp_val,
    temp_val_pred,
    y_temp_test,
    temp_test_pred,
    y_rain_val,
    rain_val_pred,
    y_rain_test,
    rain_test_pred,
) -> None:
    val_temp = temperature_prediction_table(val_df, y_temp_val, temp_val_pred)
    test_temp = temperature_prediction_table(test_df, y_temp_test, temp_test_pred)
    val_rain = rain_prediction_table(val_df, y_rain_val, rain_val_pred)
    test_rain = rain_prediction_table(test_df, y_rain_test, rain_test_pred)
    val_temp.to_csv(TABLES_DIR / "temperature_validation_predictions.csv", index=False)
    test_temp.to_csv(TABLES_DIR / "temperature_test_predictions.csv", index=False)
    val_rain.to_csv(TABLES_DIR / "rain_validation_predictions.csv", index=False)
    test_rain.to_csv(TABLES_DIR / "rain_test_predictions.csv", index=False)

    summary = pd.DataFrame(
        [
            temperature_summary("validation", val_temp),
            temperature_summary("test", test_temp),
            rain_summary("validation", val_rain),
            rain_summary("test", test_rain),
        ]
    )
    summary.to_csv(TABLES_DIR / "validation_test_prediction_summary.csv", index=False)
    return {
        "temperature_validation": val_temp,
        "temperature_test": test_temp,
        "rain_validation": val_rain,
        "rain_test": test_rain,
    }


def temperature_prediction_table(df, y_true, y_pred):
    result = df[["time"]].copy()
    result["actual_temperature"] = np.asarray(y_true)
    result["predicted_temperature"] = np.asarray(y_pred)
    result["residual"] = result["actual_temperature"] - result["predicted_temperature"]
    result["absolute_error"] = result["residual"].abs()
    for tolerance in [1, 2, 3]:
        result[f"within_{tolerance}c"] = result["absolute_error"] <= tolerance
    return result


def rain_prediction_table(df, y_true, y_pred):
    result = df[["time"]].copy()
    result["actual_rain"] = np.asarray(y_true).astype(int)
    result["predicted_rain"] = np.asarray(y_pred).astype(int)
    result["correct"] = result["actual_rain"] == result["predicted_rain"]
    return result


def temperature_summary(split, table):
    return {
        "split": split,
        "task": "temperature_regression",
        "mae": table["absolute_error"].mean(),
        "rmse": float(np.sqrt(np.mean(table["residual"] ** 2))),
        "within_1c": table["within_1c"].mean(),
        "within_2c": table["within_2c"].mean(),
        "within_3c": table["within_3c"].mean(),
    }


def rain_summary(split, table):
    y_true = table["actual_rain"]
    y_pred = table["predicted_rain"]
    return {
        "split": split,
        "task": "rain_classification",
        "correct_count": int(table["correct"].sum()),
        "wrong_count": int((~table["correct"]).sum()),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def save_worst_prediction_days(test_df: pd.DataFrame, predictions) -> None:
    errors = test_df[["time", "temperature", "target_temperature", "target_rain"]].copy()
    errors["predicted_temperature"] = predictions
    errors["absolute_error"] = np.abs(errors["target_temperature"] - errors["predicted_temperature"])
    errors.sort_values("absolute_error", ascending=False).head(20).to_csv(
        TABLES_DIR / "worst_prediction_days.csv",
        index=False,
    )


def save_split_summary(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    rows = []
    for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
        rows.append(
            {
                "split": name,
                "rows": len(df),
                "start": df["time"].min(),
                "end": df["time"].max(),
                "rain_rate": df["target_rain"].mean(),
            }
        )
    pd.DataFrame(rows).to_csv(TABLES_DIR / "split_summary.csv", index=False)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile seven_day_pipeline.py
import numpy as np
import os
import pandas as pd
import json
from matplotlib import pyplot as plt
from scipy.stats import randint, uniform
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier, XGBRegressor
except ImportError:
    XGBClassifier = None
    XGBRegressor = None

from src.config import FALLBACK_RAW_DATA_PATH, RAW_DATA_PATH, REPORTS_DIR, TABLES_DIR, ensure_directories
from src.data_loader import DataLoader
from src.preprocessing import Preprocessor
from src.feature_engineering import FeatureEngineer
from src.split import chronological_split
from src.utils import weather_level


SEVEN_DAY_DIR = REPORTS_DIR / os.environ.get("SEVEN_DAY_OUTPUT_DIR", "seven_day_forecast")
SEVEN_DAY_TABLES = SEVEN_DAY_DIR / "tables"
SEVEN_DAY_FIGURES = SEVEN_DAY_DIR / "figures"
SEARCH_ITER = int(os.environ.get("SEVEN_DAY_SEARCH_ITER", "4"))
RANDOM_STATE = 42


def main() -> None:
    ensure_directories()
    SEVEN_DAY_TABLES.mkdir(parents=True, exist_ok=True)
    SEVEN_DAY_FIGURES.mkdir(parents=True, exist_ok=True)

    csv_path = RAW_DATA_PATH if RAW_DATA_PATH.exists() else FALLBACK_RAW_DATA_PATH
    raw_df = DataLoader(csv_path).load()
    noon_df = Preprocessor().preprocess(raw_df, save=False)
    base_feature_df = FeatureEngineer().transform(noon_df)
    feature_cols = select_feature_columns(base_feature_df)

    train_df, val_df, test_df = chronological_split(base_feature_df)
    horizon_metrics = []
    selected_rows = []
    forecast_rows = []
    test_prediction_rows = []
    threshold_curve_rows = []
    hyperparameter_tuning_rows = []

    for horizon in range(1, 8):
        horizon_df = build_horizon_targets(noon_df, horizon)
        horizon_feature_df = merge_horizon_targets(base_feature_df, horizon_df)
        train_h, val_h, test_h = chronological_split(horizon_feature_df)

        X_train = train_h[feature_cols]
        y_temp_train = train_h["horizon_temperature"]
        y_rain_train = train_h["horizon_rain"].astype(int)
        X_val = val_h[feature_cols]
        X_test = test_h[feature_cols]

        temp_models, temp_tuning_rows = train_temperature_candidates(horizon, X_train, y_temp_train)
        rain_models, rain_tuning_rows = train_rain_candidates(horizon, X_train, y_rain_train)
        hyperparameter_tuning_rows.extend(temp_tuning_rows + rain_tuning_rows)
        temp_val_metrics = evaluate_temperature_candidates(horizon, "validation", val_h, X_val, temp_models)
        temp_test_metrics = evaluate_temperature_candidates(horizon, "test", test_h, X_test, temp_models)
        rain_thresholds = tune_rain_thresholds(horizon, val_h, X_val, rain_models)
        for threshold_info in rain_thresholds.values():
            threshold_curve_rows.extend(threshold_info["curve"])
        rain_val_metrics = evaluate_rain_candidates(horizon, "validation", val_h, X_val, rain_models, rain_thresholds)
        rain_test_metrics = evaluate_rain_candidates(horizon, "test", test_h, X_test, rain_models, rain_thresholds)
        horizon_metrics.extend(temp_val_metrics + temp_test_metrics + rain_val_metrics + rain_test_metrics)

        temp_val_df = pd.DataFrame(temp_val_metrics)
        rain_val_df = pd.DataFrame(rain_val_metrics)
        best_temp_model = temp_val_df.sort_values("rmse").iloc[0]["model"]
        best_rain_model = rain_val_df.sort_values("f1", ascending=False).iloc[0]["model"]
        best_temp_test_pred = temp_models[best_temp_model].predict(X_test)
        best_rain_test_prob = rain_probabilities(rain_models[best_rain_model], X_test)
        best_rain_threshold = rain_thresholds[best_rain_model]["threshold"]
        best_rain_test_pred = (best_rain_test_prob >= best_rain_threshold).astype(int)
        test_prediction_rows.extend(
            build_test_prediction_rows(
                horizon,
                test_h,
                best_temp_model,
                best_rain_model,
                best_temp_test_pred,
                best_rain_test_pred,
            )
        )
        selected_rows.append(
            {
                "horizon_day": horizon,
                "best_temperature_model": best_temp_model,
                "validation_rmse": temp_val_df[temp_val_df["model"] == best_temp_model]["rmse"].iloc[0],
                "test_rmse": pd.DataFrame(temp_test_metrics)[
                    pd.DataFrame(temp_test_metrics)["model"] == best_temp_model
                ]["rmse"].iloc[0],
                "best_rain_model": best_rain_model,
                "validation_f1": rain_val_df[rain_val_df["model"] == best_rain_model]["f1"].iloc[0],
                "test_f1": pd.DataFrame(rain_test_metrics)[
                    pd.DataFrame(rain_test_metrics)["model"] == best_rain_model
                ]["f1"].iloc[0],
                "rain_threshold": best_rain_threshold,
            }
        )

        latest_features = base_feature_df.sort_values("time").iloc[[-1]][feature_cols]
        pred_temp = float(temp_models[best_temp_model].predict(latest_features)[0])
        pred_rain_prob = float(rain_probabilities(rain_models[best_rain_model], latest_features)[0])
        pred_rain = int(pred_rain_prob >= best_rain_threshold)
        pred_date = base_feature_df["time"].max() + pd.Timedelta(days=horizon)
        forecast_rows.append(
            {
                "horizon_day": horizon,
                "forecast_time": pred_date,
                "temperature_model": best_temp_model,
                "rain_model": best_rain_model,
                "predicted_temperature": pred_temp,
                "predicted_rain": pred_rain,
                "predicted_rain_probability": pred_rain_prob,
                "rain_threshold": best_rain_threshold,
                "predicted_rain_label": "rain" if pred_rain == 1 else "no_rain",
                "predicted_weather_level": weather_level(pred_temp),
            }
        )

    metrics_df = pd.DataFrame(horizon_metrics)
    selected_df = pd.DataFrame(selected_rows)
    forecast_df = pd.DataFrame(forecast_rows)
    test_predictions_df = pd.DataFrame(test_prediction_rows)
    threshold_df = pd.DataFrame(threshold_curve_rows)
    tuning_df = pd.DataFrame(hyperparameter_tuning_rows)
    best_hyperparameters_df = build_best_hyperparameters_table(tuning_df, selected_df)
    metrics_df.to_csv(SEVEN_DAY_TABLES / "seven_day_horizon_metrics.csv", index=False)
    selected_df.to_csv(SEVEN_DAY_TABLES / "seven_day_selected_models.csv", index=False)
    forecast_df.to_csv(SEVEN_DAY_TABLES / "seven_day_forecast_predictions.csv", index=False)
    test_predictions_df.to_csv(SEVEN_DAY_TABLES / "seven_day_test_predictions.csv", index=False)
    threshold_df.to_csv(SEVEN_DAY_TABLES / "seven_day_threshold_tuning.csv", index=False)
    tuning_df.to_csv(SEVEN_DAY_TABLES / "seven_day_hyperparameter_tuning.csv", index=False)
    best_hyperparameters_df.to_csv(SEVEN_DAY_TABLES / "seven_day_best_hyperparameters.csv", index=False)

    plot_metrics(metrics_df)
    plot_model_comparisons(metrics_df)
    plot_test_actual_vs_predicted(test_predictions_df)
    plot_threshold_tuning(threshold_df, selected_df)
    plot_hyperparameter_tuning(tuning_df, selected_df)
    plot_forecast(forecast_df)
    write_report(raw_df, noon_df, base_feature_df, metrics_df, selected_df, forecast_df, best_hyperparameters_df)
    print("7-day forecast pipeline completed successfully.")


def select_feature_columns(df: pd.DataFrame) -> list[str]:
    blocked = {
        "time",
        "target_temperature",
        "target_precipitation",
        "target_rain",
        "target_weather_level",
    }
    return [c for c in df.select_dtypes(include="number").columns if c not in blocked]


def build_horizon_targets(noon_df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    df = noon_df[["time", "temperature", "precipitation"]].copy().sort_values("time")
    df["horizon_temperature"] = df["temperature"].shift(-horizon)
    df["horizon_precipitation"] = df["precipitation"].shift(-horizon)
    df["horizon_rain"] = (df["horizon_precipitation"] > 0).astype(float)
    return df[["time", "horizon_temperature", "horizon_precipitation", "horizon_rain"]]


def merge_horizon_targets(base_feature_df: pd.DataFrame, horizon_df: pd.DataFrame) -> pd.DataFrame:
    df = base_feature_df.drop(
        columns=["target_temperature", "target_precipitation", "target_rain", "target_weather_level"],
        errors="ignore",
    ).merge(horizon_df, on="time", how="inner")
    df = df.dropna(subset=["horizon_temperature", "horizon_precipitation", "horizon_rain"])
    return df.reset_index(drop=True)


def train_temperature_candidates(horizon, X_train, y_train) -> tuple[dict, list[dict]]:
    models = {
        "LinearRegression": Pipeline(
            [("scaler", StandardScaler()), ("model", LinearRegression())]
        ),
        "KNNRegressor": Pipeline(
            [("scaler", StandardScaler()), ("model", KNeighborsRegressor())]
        ),
    }
    searches = {
        "KNNRegressor": {
            "model__n_neighbors": randint(3, 26),
            "model__weights": ["uniform", "distance"],
        }
    }
    if XGBRegressor is not None:
        models["XGBoostRegressor"] = XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
        )
        searches["XGBoostRegressor"] = {
            "n_estimators": randint(100, 501),
            "max_depth": randint(3, 11),
            "learning_rate": uniform(0.01, 0.29),
        }

    cv = TimeSeriesSplit(n_splits=5)
    fitted = {}
    tuning_rows = []
    for name, model in models.items():
        if name in searches:
            model = RandomizedSearchCV(
                model,
                searches[name],
                n_iter=SEARCH_ITER,
                cv=cv,
                scoring="neg_root_mean_squared_error",
                random_state=RANDOM_STATE,
                n_jobs=1,
            )
        model.fit(X_train, y_train)
        fitted[name] = model
        tuning_rows.extend(extract_tuning_rows(horizon, "temperature_regression", name, model))
    return fitted, tuning_rows


def train_rain_candidates(horizon, X_train, y_train) -> tuple[dict, list[dict]]:
    y_train = y_train.astype(int)
    models = {
        "LogisticRegression": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "SVM": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "model",
                    SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE),
                ),
            ]
        ),
    }
    searches = {
        "LogisticRegression": {"model__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
        "SVM": {"model__C": [0.1, 1.0, 10.0, 100.0], "model__kernel": ["linear", "rbf"]},
    }
    if XGBClassifier is not None:
        neg = max(1, int((y_train == 0).sum()))
        pos = max(1, int((y_train == 1).sum()))
        models["XGBoostClassifier"] = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=neg / pos,
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
        )
        searches["XGBoostClassifier"] = {
            "n_estimators": randint(100, 501),
            "max_depth": randint(3, 11),
            "learning_rate": uniform(0.01, 0.29),
        }
    cv = TimeSeriesSplit(n_splits=5)
    fitted = {}
    tuning_rows = []
    for name, model in models.items():
        n_iter = min(SEARCH_ITER, search_space_size(searches[name]))
        model = RandomizedSearchCV(
            model,
            searches[name],
            n_iter=n_iter,
            cv=cv,
            scoring="f1",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
        model.fit(X_train, y_train)
        fitted[name] = model
        tuning_rows.extend(extract_tuning_rows(horizon, "rain_classification", name, model))
    return fitted, tuning_rows


def search_space_size(param_distributions) -> int:
    for values in param_distributions.values():
        if not isinstance(values, (list, tuple)):
            return SEARCH_ITER
    return max(1, len(list(ParameterGrid(param_distributions))))


def extract_tuning_rows(horizon: int, task: str, model_name: str, fitted_model) -> list[dict]:
    if not hasattr(fitted_model, "cv_results_"):
        return [
            {
                "horizon_day": horizon,
                "task": task,
                "model": model_name,
                "candidate_index": 1,
                "rank_test_score": 1,
                "mean_test_score": np.nan,
                "std_test_score": np.nan,
                "mean_cv_rmse": np.nan,
                "mean_cv_f1": np.nan,
                "is_best": True,
                "best_params": describe_best_params(fitted_model),
                "params": describe_best_params(fitted_model),
                "search_iter_budget": SEARCH_ITER,
            }
        ]

    results = fitted_model.cv_results_
    best_params = describe_best_params(fitted_model)
    rows = []
    for idx, params in enumerate(results["params"]):
        score = float(results["mean_test_score"][idx])
        row = {
            "horizon_day": horizon,
            "task": task,
            "model": model_name,
            "candidate_index": idx + 1,
            "rank_test_score": int(results["rank_test_score"][idx]),
            "mean_test_score": score,
            "std_test_score": float(results["std_test_score"][idx]),
            "mean_cv_rmse": -score if task == "temperature_regression" else np.nan,
            "mean_cv_f1": score if task == "rain_classification" else np.nan,
            "is_best": int(results["rank_test_score"][idx]) == 1,
            "best_params": best_params,
            "params": json.dumps(params, ensure_ascii=False, sort_keys=True),
            "search_iter_budget": SEARCH_ITER,
        }
        rows.append(row)
    return rows


def describe_best_params(model) -> str:
    if hasattr(model, "best_params_"):
        return json.dumps(model.best_params_, ensure_ascii=False, sort_keys=True)
    if isinstance(model, Pipeline):
        return "StandardScaler + LinearRegression"
    return model.__class__.__name__


def build_best_hyperparameters_table(tuning_df: pd.DataFrame, selected_df: pd.DataFrame) -> pd.DataFrame:
    if tuning_df.empty:
        return pd.DataFrame()
    best = tuning_df[tuning_df["is_best"] == True].copy()
    best = best[
        [
            "horizon_day",
            "task",
            "model",
            "rank_test_score",
            "mean_cv_rmse",
            "mean_cv_f1",
            "best_params",
            "search_iter_budget",
        ]
    ].drop_duplicates()

    selected_records = []
    for _, row in selected_df.iterrows():
        selected_records.append(
            {
                "horizon_day": row["horizon_day"],
                "task": "temperature_regression",
                "selected_model": row["best_temperature_model"],
                "selected_metric": "validation_rmse",
                "selected_metric_value": row["validation_rmse"],
            }
        )
        selected_records.append(
            {
                "horizon_day": row["horizon_day"],
                "task": "rain_classification",
                "selected_model": row["best_rain_model"],
                "selected_metric": "validation_f1",
                "selected_metric_value": row["validation_f1"],
            }
        )
    selected = pd.DataFrame(selected_records)
    return best.merge(
        selected,
        left_on=["horizon_day", "task", "model"],
        right_on=["horizon_day", "task", "selected_model"],
        how="left",
    )


def evaluate_temperature_candidates(horizon, split, df, X, models) -> list[dict]:
    rows = []
    for name, model in models.items():
        pred = model.predict(X)
        rows.append(evaluate_temperature(horizon, split, name, df, pred, model))
    return rows


def tune_rain_thresholds(horizon, val_df, X_val, models) -> dict:
    y_true = val_df["horizon_rain"].astype(int).to_numpy()
    thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
    tuned = {}
    for model_name, model in models.items():
        probs = rain_probabilities(model, X_val)
        rows = []
        best = {"threshold": 0.5, "f1": -1.0}
        for threshold in thresholds:
            pred = (probs >= threshold).astype(int)
            row = {
                "horizon_day": horizon,
                "model": model_name,
                "threshold": threshold,
                "accuracy": accuracy_score(y_true, pred),
                "precision": precision_score(y_true, pred, zero_division=0),
                "recall": recall_score(y_true, pred, zero_division=0),
                "f1": f1_score(y_true, pred, zero_division=0),
            }
            rows.append(row)
            if row["f1"] > best["f1"]:
                best = {"threshold": threshold, "f1": row["f1"]}
        tuned[model_name] = {"threshold": best["threshold"], "validation_f1": best["f1"], "curve": rows}
    return tuned


def rain_probabilities(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    return model.predict(X)


def evaluate_rain_candidates(horizon, split, df, X, models, thresholds) -> list[dict]:
    rows = []
    for name, model in models.items():
        probs = rain_probabilities(model, X)
        threshold = thresholds[name]["threshold"]
        pred = (probs >= threshold).astype(int)
        rows.append(evaluate_rain(horizon, split, name, df, pred, threshold, model))
    return rows


def evaluate_temperature(horizon, split, model_name, df, temp_pred, model) -> dict:
    y_temp = df["horizon_temperature"]
    abs_error = np.abs(y_temp - temp_pred)
    return {
        "horizon_day": horizon,
        "split": split,
        "task": "temperature_regression",
        "model": model_name,
        "best_params": describe_best_params(model),
        "search_space": model_params(model_name),
        "mae": mean_absolute_error(y_temp, temp_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_temp, temp_pred))),
        "r2": r2_score(y_temp, temp_pred),
        "within_1c": float((abs_error <= 1).mean()),
        "within_2c": float((abs_error <= 2).mean()),
        "within_3c": float((abs_error <= 3).mean()),
    }


def evaluate_rain(horizon, split, model_name, df, rain_pred, threshold, model) -> dict:
    y_rain = df["horizon_rain"].astype(int)
    return {
        "horizon_day": horizon,
        "split": split,
        "task": "rain_classification",
        "model": model_name,
        "best_params": describe_best_params(model),
        "search_space": model_params(model_name),
        "threshold": threshold,
        "accuracy": accuracy_score(y_rain, rain_pred),
        "precision": precision_score(y_rain, rain_pred, zero_division=0),
        "recall": recall_score(y_rain, rain_pred, zero_division=0),
        "f1": f1_score(y_rain, rain_pred, zero_division=0),
    }


def build_test_prediction_rows(
    horizon: int,
    test_df: pd.DataFrame,
    temp_model_name: str,
    rain_model_name: str,
    temp_pred,
    rain_pred,
) -> list[dict]:
    rows = []
    for i, (_, row) in enumerate(test_df.reset_index(drop=True).iterrows()):
        actual_temp = row["horizon_temperature"]
        predicted_temp = float(temp_pred[i])
        actual_rain = int(row["horizon_rain"])
        predicted_rain = int(rain_pred[i])
        rows.append(
            {
                "horizon_day": horizon,
                "time": row["time"],
                "temperature_model": temp_model_name,
                "rain_model": rain_model_name,
                "actual_temperature": actual_temp,
                "predicted_temperature": predicted_temp,
                "temperature_residual": actual_temp - predicted_temp,
                "actual_rain": actual_rain,
                "predicted_rain": predicted_rain,
                "rain_correct": actual_rain == predicted_rain,
            }
        )
    return rows


def model_params(model_name: str) -> str:
    if model_name == "LinearRegression":
        return "StandardScaler + LinearRegression"
    if model_name == "KNNRegressor":
        return "RandomizedSearchCV: n_neighbors 3-25, weights uniform/distance"
    if model_name == "XGBoostRegressor":
        return "RandomizedSearchCV: n_estimators 100-500, max_depth 3-10, learning_rate 0.01-0.3"
    if model_name == "LogisticRegression":
        return "RandomizedSearchCV: C in [0.01, 0.1, 1, 10, 100], class_weight balanced"
    if model_name == "SVM":
        return "RandomizedSearchCV: C in [0.1, 1, 10, 100], kernel linear/rbf, class_weight balanced"
    if model_name == "XGBoostClassifier":
        return "RandomizedSearchCV: n_estimators 100-500, max_depth 3-10, learning_rate 0.01-0.3, scale_pos_weight"
    return ""


def plot_metrics(metrics_df: pd.DataFrame) -> None:
    temp = metrics_df[(metrics_df["task"] == "temperature_regression") & (metrics_df["split"] == "test")]
    rain = metrics_df[(metrics_df["task"] == "rain_classification") & (metrics_df["split"] == "test")]

    plt.figure(figsize=(8, 5))
    for model in temp["model"].unique():
        model_df = temp[temp["model"] == model]
        plt.plot(model_df["horizon_day"], model_df["rmse"], marker="o", label=model)
    plt.title("7-day temperature RMSE by horizon and model")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("RMSE (degC)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "seven_day_temperature_error_by_horizon.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    for model in rain["model"].unique():
        model_df = rain[rain["model"] == model]
        plt.plot(model_df["horizon_day"], model_df["f1"], marker="o", label=model)
    plt.title("7-day rain F1 by horizon and model")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("F1")
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "seven_day_rain_performance_by_horizon.png", dpi=160)
    plt.close()


def plot_model_comparisons(metrics_df: pd.DataFrame) -> None:
    test = metrics_df[metrics_df["split"] == "test"]
    temp = test[test["task"] == "temperature_regression"]
    rain = test[test["task"] == "rain_classification"]

    plt.figure(figsize=(11, 5))
    temp_pivot = temp.pivot(index="horizon_day", columns="model", values="rmse")
    temp_pivot.plot(kind="line", marker="o", ax=plt.gca())
    plt.title("Test RMSE comparison for 7-day temperature models")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("RMSE (degC)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "seven_day_temperature_model_comparison.png", dpi=160)
    plt.close()

    plt.figure(figsize=(11, 5))
    rain_pivot = rain.pivot(index="horizon_day", columns="model", values="f1")
    rain_pivot.plot(kind="line", marker="o", ax=plt.gca())
    plt.title("Test F1 comparison for 7-day rain models")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("F1")
    plt.ylim(0, 1)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "seven_day_rain_model_comparison.png", dpi=160)
    plt.close()


def plot_hyperparameter_tuning(tuning_df: pd.DataFrame, selected_df: pd.DataFrame) -> None:
    if tuning_df.empty:
        return

    tuned = tuning_df.dropna(subset=["rank_test_score"]).copy()
    if tuned.empty:
        return

    for (task, model), df in tuned.groupby(["task", "model"]):
        metric_col = "mean_cv_rmse" if task == "temperature_regression" else "mean_cv_f1"
        ylabel = "Mean CV RMSE (lower is better)" if task == "temperature_regression" else "Mean CV F1 (higher is better)"
        selected_model_col = "best_temperature_model" if task == "temperature_regression" else "best_rain_model"

        fig, axes = plt.subplots(7, 1, figsize=(13, 24), sharex=False)
        for ax, horizon in zip(axes, range(1, 8)):
            h_df = df[df["horizon_day"] == horizon].sort_values("candidate_index")
            if h_df.empty:
                ax.set_visible(False)
                continue
            ax.plot(h_df["candidate_index"], h_df[metric_col], marker="o", linewidth=1.2)
            best = h_df.sort_values("rank_test_score").iloc[0]
            ax.scatter([best["candidate_index"]], [best[metric_col]], color="red", zorder=3)
            selected = selected_df[selected_df["horizon_day"] == horizon][selected_model_col].iloc[0]
            is_selected = "selected" if selected == model else "not selected"
            best_params = shorten_params(str(best["best_params"]))
            ax.set_title(
                f"Horizon {horizon}: {model} tuning ({is_selected}); best params: {best_params}",
                fontsize=9,
            )
            ax.set_ylabel(ylabel)
            ax.grid(alpha=0.25)
        axes[-1].set_xlabel("RandomizedSearchCV candidate index")
        plt.tight_layout()
        filename = f"hyperparameter_tuning_{task}_{model}.png"
        plt.savefig(SEVEN_DAY_FIGURES / filename, dpi=160)
        plt.close()

    selected_tuned = tuned[tuned["is_best"] == True].copy()
    for task, df in selected_tuned.groupby("task"):
        metric_col = "mean_cv_rmse" if task == "temperature_regression" else "mean_cv_f1"
        ylabel = "Best mean CV RMSE" if task == "temperature_regression" else "Best mean CV F1"
        plt.figure(figsize=(11, 5))
        for model in df["model"].unique():
            model_df = df[df["model"] == model].sort_values("horizon_day")
            plt.plot(model_df["horizon_day"], model_df[metric_col], marker="o", label=model)
        title = "Best tuned CV score by horizon for temperature models" if task == "temperature_regression" else "Best tuned CV score by horizon for rain models"
        plt.title(title)
        plt.xlabel("Forecast horizon day")
        plt.ylabel(ylabel)
        if task == "rain_classification":
            plt.ylim(0, 1)
        plt.legend()
        plt.tight_layout()
        plt.savefig(SEVEN_DAY_FIGURES / f"hyperparameter_tuning_best_scores_{task}.png", dpi=160)
        plt.close()


def shorten_params(params: str, max_len: int = 130) -> str:
    return params if len(params) <= max_len else params[: max_len - 3] + "..."


def plot_test_actual_vs_predicted(test_predictions_df: pd.DataFrame) -> None:
    horizons = sorted(test_predictions_df["horizon_day"].unique())
    fig, axes = plt.subplots(7, 1, figsize=(13, 22), sharex=False)
    for ax, horizon in zip(axes, horizons):
        df = test_predictions_df[test_predictions_df["horizon_day"] == horizon].copy()
        df["time"] = pd.to_datetime(df["time"])
        ax.plot(df["time"], df["actual_temperature"], label="Actual", linewidth=1.2)
        ax.plot(df["time"], df["predicted_temperature"], label="Predicted", linewidth=1.2)
        model_name = df["temperature_model"].iloc[0]
        ax.set_title(f"Horizon {horizon}: actual vs predicted temperature on test set ({model_name})")
        ax.set_ylabel("degC")
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("Test date")
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "test_actual_vs_predicted_temperature_best_models_by_horizon.png", dpi=160)
    plt.close()

    plt.figure(figsize=(13, 6))
    h1 = test_predictions_df[test_predictions_df["horizon_day"] == 1].copy()
    h1["time"] = pd.to_datetime(h1["time"])
    plt.plot(h1["time"], h1["actual_temperature"], label="Actual", linewidth=1.3)
    plt.plot(h1["time"], h1["predicted_temperature"], label="Predicted", linewidth=1.3)
    plt.title("Horizon 1 actual vs predicted temperature on test set")
    plt.xlabel("Test date")
    plt.ylabel("Temperature (degC)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "test_actual_vs_predicted_temperature_horizon_1.png", dpi=160)
    plt.close()

    fig, axes = plt.subplots(7, 1, figsize=(13, 22), sharex=False)
    for ax, horizon in zip(axes, horizons):
        df = test_predictions_df[test_predictions_df["horizon_day"] == horizon].copy()
        df["time"] = pd.to_datetime(df["time"])
        ax.step(df["time"], df["actual_rain"], label="Actual rain", linewidth=1.2, where="mid")
        ax.step(df["time"], df["predicted_rain"], label="Predicted rain", linewidth=1.2, where="mid")
        model_name = df["rain_model"].iloc[0]
        ax.set_title(f"Horizon {horizon}: actual vs predicted rain on test set ({model_name})")
        ax.set_ylabel("Rain label")
        ax.set_ylim(-0.1, 1.1)
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("Test date")
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "test_actual_vs_predicted_rain_best_models_by_horizon.png", dpi=160)
    plt.close()

    rows = []
    for horizon in horizons:
        df = test_predictions_df[test_predictions_df["horizon_day"] == horizon]
        rows.append(
            {
                "horizon_day": horizon,
                "correct_rate": df["rain_correct"].mean(),
                "wrong_rate": 1 - df["rain_correct"].mean(),
            }
        )
    rate_df = pd.DataFrame(rows).melt(
        id_vars="horizon_day",
        value_vars=["correct_rate", "wrong_rate"],
        var_name="result",
        value_name="rate",
    )
    plt.figure(figsize=(10, 5))
    for result in ["correct_rate", "wrong_rate"]:
        part = rate_df[rate_df["result"] == result]
        plt.plot(part["horizon_day"], part["rate"], marker="o", label=result)
    plt.title("Rain prediction correct/wrong rate on test set by horizon")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("Rate")
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "test_rain_correct_wrong_rate_by_horizon.png", dpi=160)
    plt.close()


def plot_threshold_tuning(threshold_df: pd.DataFrame, selected_df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(7, 1, figsize=(11, 22), sharex=True)
    for ax, (_, selected) in zip(axes, selected_df.iterrows()):
        horizon = selected["horizon_day"]
        model = selected["best_rain_model"]
        threshold = selected["rain_threshold"]
        df = threshold_df[
            (threshold_df["horizon_day"] == horizon)
            & (threshold_df["model"] == model)
        ]
        ax.plot(df["threshold"], df["f1"], marker="o", label=f"{model} validation F1")
        ax.axvline(threshold, color="red", linestyle="--", label=f"best threshold={threshold:.2f}")
        ax.set_title(f"Horizon {horizon}: threshold tuning for selected rain model")
        ax.set_ylabel("F1")
        ax.set_ylim(0, 1)
        ax.legend(loc="lower right")
    axes[-1].set_xlabel("Decision threshold")
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "rain_threshold_tuning_by_horizon.png", dpi=160)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(selected_df["horizon_day"], selected_df["rain_threshold"], marker="o")
    plt.title("Selected rain decision threshold by horizon")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("Selected threshold")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "selected_rain_threshold_by_horizon.png", dpi=160)
    plt.close()

def plot_forecast(forecast_df: pd.DataFrame) -> None:
    plt.figure(figsize=(9, 5))
    plt.plot(forecast_df["forecast_time"], forecast_df["predicted_temperature"], marker="o")
    plt.title("Predicted noon temperature for the next 7 days")
    plt.xlabel("Forecast date")
    plt.ylabel("Predicted temperature (degC)")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "next_7_days_temperature_forecast.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.bar(forecast_df["horizon_day"], forecast_df["predicted_rain"])
    plt.title("Predicted rain label for the next 7 days")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("Rain prediction (1=rain, 0=no rain)")
    plt.ylim(0, 1.2)
    plt.tight_layout()
    plt.savefig(SEVEN_DAY_FIGURES / "next_7_days_rain_forecast.png", dpi=160)
    plt.close()


def write_report(raw_df, noon_df, feature_df, metrics_df, selected_df, forecast_df, best_hyperparameters_df) -> None:
    validation_temp = metrics_df[(metrics_df["task"] == "temperature_regression") & (metrics_df["split"] == "validation")]
    validation_rain = metrics_df[(metrics_df["task"] == "rain_classification") & (metrics_df["split"] == "validation")]
    test_temp = metrics_df[(metrics_df["task"] == "temperature_regression") & (metrics_df["split"] == "test")]
    test_rain = metrics_df[(metrics_df["task"] == "rain_classification") & (metrics_df["split"] == "test")]
    stats = noon_df.select_dtypes(include="number").describe().round(3).reset_index()
    content = f"""# Hanoi 7-Day Noon Weather Forecasting Report

## Introduction
This report presents an extended version of the original Hanoi noon weather prediction system. Instead of predicting only the next day, this version predicts noon temperature and rain for the next 7 days. The problem is formulated as a multi-horizon supervised learning task.

## Motivation
Short-term weather forecasting is useful for planning travel, outdoor work, school activities, and daily scheduling. A 7-day forecast is more practical than a single next-day prediction because users often need to plan several days ahead. However, forecasting difficulty increases with horizon length because future atmospheric conditions become less certain.

## Contribution
This version contributes a separate 7-day forecasting pipeline with the following components:

- Multi-horizon target generation for day 1 to day 7.
- Temperature regression model for each horizon.
- Rain classification model for each horizon.
- Chronological train/validation/test evaluation.
- Horizon-wise metrics and figures.
- Final 7-day forecast output from the latest available noon observation.

## Project Objective
The objective is to predict Hanoi weather at 12:00 PM for each of the next 7 days.

The two machine learning tasks are:

- Temperature regression: predict noon temperature at horizon `h`, where `h = 1..7`.
- Rain classification: predict whether noon precipitation at horizon `h` is greater than 0.

Weather level is not trained as a separate model. It is derived from predicted temperature:

```text
temperature < 22      -> cool
22 <= temperature <=30 -> normal
temperature > 30      -> hot
```

## Dataset
- Raw hourly records: {len(raw_df):,}
- Noon records: {len(noon_df):,}
- Engineered supervised samples before horizon-specific target filtering: {len(feature_df):,}
- Latest available noon record used for next-7-day inference: {feature_df['time'].max()}

The data contains historical Open-Meteo observations from 2020 to 2025. Only records at 12:00 are used to keep the forecasting target consistent with the project objective.

## Dataset Statistics

{to_markdown(stats)}

The average noon temperature is about 26.9°C, while precipitation has a median of 0.0 mm. This indicates that rainfall is sparse at noon, making rain classification more difficult than temperature regression.

## Input and Expected Output
Input:

```text
Historical 12:00 weather records with engineered lag, rolling, and time features.
```

Outputs:

```text
For each horizon h = 1..7:
- predicted noon temperature
- predicted rain / no rain label
- derived weather level
```

## Data Preprocessing
The preprocessing logic follows the original project:

1. Detect and remove Open-Meteo metadata rows.
2. Normalize weather column names.
3. Parse the `time` column as datetime.
4. Sort observations chronologically.
5. Remove duplicated timestamps.
6. Keep only records where `hour == 12`.
7. Remove unused or leakage-prone columns.
8. Fill numeric missing values with medians if needed.

## Feature Engineering
The 7-day pipeline reuses the same feature engineering design as the original system:

- Time features: `month`, `day_of_year`, `season`.
- Lag features for temperature, humidity, and precipitation at lags 1, 3, and 7.
- Rolling features using shifted rolling windows.

Rolling features are computed with `shift(1)` before `rolling()`. This is important because the model must not use information from the day being predicted.

## Methodology
For each horizon `h = 1..7`, the target is generated as:

```text
horizon_temperature = temperature.shift(-h)
horizon_rain = precipitation.shift(-h) > 0
```

This means that horizon 1 predicts tomorrow, horizon 2 predicts two days later, and so on until horizon 7.

The same input feature set is used for each horizon, but each horizon has its own target and its own trained model pair. This direct multi-horizon approach avoids recursive error accumulation.

## Train / Validation / Test Protocol
The data is split chronologically:

| Split | Period | Role |
|---|---|---|
| Training | 2020-2023 | Fit model parameters |
| Validation | 2024 | Check model behavior and compare horizons |
| Test | 2025 | Final evaluation |

The test set is not used to train or select models. This preserves a realistic evaluation on unseen future data.

## Model Training Process
This version trains and compares multiple candidate models for each horizon:

Temperature regression candidates:

- Linear Regression.
- KNN Regressor.
- XGBoost Regressor.

Rain classification candidates:

- Logistic Regression with balanced class weights.
- SVM with balanced class weights.
- XGBoost Classifier with automatic `scale_pos_weight`.

For each horizon, the temperature model uses a pipeline:

```text
StandardScaler -> LinearRegression / KNN
XGBoostRegressor without scaling
```

Linear Regression is fitted directly after scaling. KNN and XGBoost are tuned with `RandomizedSearchCV` and `TimeSeriesSplit`.

For rain classification, the model uses:

```text
StandardScaler -> LogisticRegression / SVM
XGBoostClassifier without scaling
```

The best temperature model for each horizon is selected by validation RMSE. The best rain model for each horizon is selected by validation F1. Test metrics are reported only after model selection.

Hyperparameter search configuration:

```text
RandomizedSearchCV n_iter = {SEARCH_ITER}
Cross-validation = TimeSeriesSplit(n_splits=5)
Regression scoring = negative RMSE
Classification scoring = F1
```

Hyperparameter tuning is performed only on the training period through time-series cross-validation. The validation period is then used to select the final model per horizon. This separation is methodologically important: cross-validation estimates which configuration fits the training history best, while validation evaluates whether that fitted model generalizes to the next chronological year.

## Best Hyperparameters from RandomizedSearchCV

{to_markdown(best_hyperparameters_df)}

The table above reports the best configuration found inside the search space for each horizon and model. Linear Regression has no randomized hyperparameter search because it is fitted by ordinary least squares after scaling. KNN tuning mainly controls the number of neighbors and the weighting rule. XGBoost tuning controls tree complexity and learning dynamics through `n_estimators`, `max_depth`, and `learning_rate`. Logistic Regression and SVM tune regularization strength through `C`; SVM additionally tunes the kernel. For rain classification, the model hyperparameters are selected before the decision threshold is tuned on the validation set.

## Selected Models by Horizon

{to_markdown(selected_df)}

## Validation Performance by Horizon

### Temperature Model Comparison on Validation Set

{to_markdown(validation_temp[['horizon_day', 'model', 'mae', 'rmse', 'r2', 'within_1c', 'within_2c', 'within_3c']])}

### Rain Model Comparison on Validation Set

{to_markdown(validation_rain[['horizon_day', 'model', 'accuracy', 'precision', 'recall', 'f1']])}

## Test Performance by Horizon

### Temperature Model Comparison on Test Set

{to_markdown(test_temp[['horizon_day', 'model', 'mae', 'rmse', 'r2', 'within_1c', 'within_2c', 'within_3c']])}

Temperature error increases as the horizon becomes longer. This is expected because the model predicts further into the future with the same historical input information.

### Rain Model Comparison on Test Set

{to_markdown(test_rain[['horizon_day', 'model', 'accuracy', 'precision', 'recall', 'f1']])}

Rain prediction performance is less stable than temperature prediction. Rainfall is sparse and less continuous, so F1 fluctuates across horizons.

## Next 7-Day Forecast Output

{to_markdown(forecast_df)}

The forecast starts from the latest available noon observation in the dataset. All predicted rain labels are `no_rain`, while predicted temperature gradually moves from normal to cool conditions across the first several days.

## Figures

- ![Temperature error by horizon](figures/seven_day_temperature_error_by_horizon.png)
- ![Rain performance by horizon](figures/seven_day_rain_performance_by_horizon.png)
- ![Temperature model comparison](figures/seven_day_temperature_model_comparison.png)
- ![Rain model comparison](figures/seven_day_rain_model_comparison.png)
- ![Temperature hyperparameter tuning best scores](figures/hyperparameter_tuning_best_scores_temperature_regression.png)
- ![Rain hyperparameter tuning best scores](figures/hyperparameter_tuning_best_scores_rain_classification.png)
- ![KNN hyperparameter tuning](figures/hyperparameter_tuning_temperature_regression_KNNRegressor.png)
- ![XGBoost Regressor hyperparameter tuning](figures/hyperparameter_tuning_temperature_regression_XGBoostRegressor.png)
- ![Logistic Regression hyperparameter tuning](figures/hyperparameter_tuning_rain_classification_LogisticRegression.png)
- ![SVM hyperparameter tuning](figures/hyperparameter_tuning_rain_classification_SVM.png)
- ![XGBoost Classifier hyperparameter tuning](figures/hyperparameter_tuning_rain_classification_XGBoostClassifier.png)
- ![Test actual vs predicted temperature by horizon](figures/test_actual_vs_predicted_temperature_best_models_by_horizon.png)
- ![Test actual vs predicted temperature horizon 1](figures/test_actual_vs_predicted_temperature_horizon_1.png)
- ![Test actual vs predicted rain by horizon](figures/test_actual_vs_predicted_rain_best_models_by_horizon.png)
- ![Test rain correct wrong rate by horizon](figures/test_rain_correct_wrong_rate_by_horizon.png)
- ![Rain threshold tuning by horizon](figures/rain_threshold_tuning_by_horizon.png)
- ![Selected rain threshold by horizon](figures/selected_rain_threshold_by_horizon.png)
- ![Next 7 days temperature forecast](figures/next_7_days_temperature_forecast.png)
- ![Next 7 days rain forecast](figures/next_7_days_rain_forecast.png)

## Data Visualization Interpretation
The temperature error plot compares RMSE across Linear Regression, KNN Regressor, and XGBoost Regressor for horizons 1 to 7. The best model is the one with the lowest validation RMSE for each horizon.

The rain performance plot compares F1 across Logistic Regression, SVM, and XGBoost Classifier. F1 is more informative than accuracy because the rain label is imbalanced. A decline or fluctuation in F1 indicates that the model has difficulty consistently identifying rainy days several days ahead.

The next-7-day temperature forecast plot visualizes the predicted cooling pattern from 2025-12-31 to 2026-01-06. The rain forecast plot indicates that the model predicts no rain for all seven future noon timestamps.

The test actual-vs-predicted temperature plot compares the best selected model's predicted values with true test values across the full 2025 test period. Each subplot corresponds to one forecast horizon. When the predicted curve follows the actual curve closely, the model captures the temporal temperature pattern well. Wider gaps indicate larger forecast errors.

The test actual-vs-predicted rain plot compares the binary rain labels across the full test period. Because rain is represented as 0/1, the plot should be interpreted as event matching rather than continuous magnitude matching. The correct/wrong rate plot summarizes how often the selected rain model matches the actual test label for each horizon.

The threshold tuning plots show how validation F1 changes as the rain decision threshold varies. Instead of using the default 0.5 threshold, the pipeline selects the threshold that maximizes validation F1 for each selected rain model and horizon. Lower thresholds usually increase recall but may reduce precision; higher thresholds usually improve precision but may miss more rain events.

The hyperparameter tuning plots show the cross-validation score obtained by each sampled configuration. The red marker identifies the configuration selected by `RandomizedSearchCV`. In regression plots, lower RMSE is better. In classification plots, higher F1 is better. Comparing these plots across models helps distinguish two stages of the experiment: first, finding the best configuration within each candidate model; second, comparing the best configured models against each other on the validation set.

## Evaluation and Discussion
The selected-model table shows which model is best at each horizon according to validation metrics. This is important because the model that performs best for day 1 does not necessarily remain best for day 7.

For temperature, model quality is judged by RMSE. Lower RMSE means lower average prediction error with stronger penalty for large errors. For rain, model quality is judged by F1, which balances precision and recall for the rainy class.

The 7-day output should therefore be interpreted as an academic multi-horizon forecasting result, not as a production-grade weather forecast. A real deployment would require future forecast covariates from numerical weather prediction models.

## Discussion
Temperature forecasting remains relatively stable across the first several horizons because temperature has strong temporal continuity and seasonal structure. However, error generally increases as the horizon becomes longer, which is expected in multi-step forecasting.

Rain forecasting is harder than temperature forecasting because rainfall is sparse and less continuous. F1 can fluctuate across horizons, indicating that rain/no-rain separation depends strongly on short-term atmospheric conditions that are not fully captured by historical noon observations alone.

## Limitations
The 7-day forecast uses historical observed features available at the latest noon record. For future days, the pipeline predicts horizon-specific targets directly rather than using real future meteorological forecast variables. A production-grade 7-day weather forecast should incorporate numerical weather prediction inputs such as future humidity, pressure, wind, cloud cover, and radiation forecasts.

Additional limitations:

- The system uses one location only.
- It does not ingest future meteorological forecasts.
- The same feature snapshot is used to infer all seven horizons.
- Rain prediction is sensitive to class imbalance and year-to-year variation.
- The models are intentionally simple for interpretability.

## Conclusion
This 7-day version provides an interpretable multi-horizon forecasting baseline for Hanoi noon temperature and rain prediction. It is suitable for academic comparison and reporting, but future work should add real forecast covariates and probabilistic rain calibration.

## Future Work
Future work should include:

- Numerical weather prediction features for future days.
- More locations across Hanoi and northern Vietnam.
- Probabilistic rain forecasts instead of only binary labels.
- Deep learning or time-series models for multi-step forecasting.
- Calibration and uncertainty intervals for temperature prediction.
"""
    (SEVEN_DAY_DIR / "final_results_7day.md").write_text(content, encoding="utf-8")


def to_markdown(df: pd.DataFrame) -> str:
    rounded = df.copy()
    for col in rounded.select_dtypes(include="number").columns:
        rounded[col] = rounded[col].round(3)
    cols = [str(c) for c in rounded.columns]
    rows = ["| " + " | ".join(cols) + " |"]
    rows.append("| " + " | ".join(["---"] * len(cols)) + " |")
    for _, row in rounded.iterrows():
        rows.append("| " + " | ".join(str(row[c]) for c in rounded.columns) + " |")
    return "\n".join(rows)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/generate_next_day_line_comparison_charts.py
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


ROOT = Path(__file__).resolve().parents[1]
REPORTS = ROOT / "reports"
TABLES = REPORTS / "tables"
FIGURES = REPORTS / "figures"


def plot_regression_line() -> None:
    df = pd.read_csv(TABLES / "regression_results.csv")
    metrics = ["mae", "rmse", "r2"]
    long_df = df.melt(id_vars="model", value_vars=metrics, var_name="metric", value_name="value")

    plt.figure(figsize=(10, 5))
    for model in long_df["model"].unique():
        part = long_df[long_df["model"] == model]
        plt.plot(part["metric"], part["value"], marker="o", linewidth=1.8, label=model)
    plt.title("Next-day temperature regression model comparison")
    plt.xlabel("Evaluation metric")
    plt.ylabel("Metric value")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / "regression_model_comparison_line.png", dpi=160)
    plt.close()


def plot_classification_line() -> None:
    df = pd.read_csv(TABLES / "rain_classification_results.csv")
    metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    long_df = df.melt(id_vars="model", value_vars=metrics, var_name="metric", value_name="value")

    plt.figure(figsize=(11, 5))
    for model in long_df["model"].unique():
        part = long_df[long_df["model"] == model]
        plt.plot(part["metric"], part["value"], marker="o", linewidth=1.8, label=model)
    plt.title("Next-day rain classification model comparison")
    plt.xlabel("Evaluation metric")
    plt.ylabel("Metric value")
    plt.ylim(0, 1)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / "classification_model_comparison_line.png", dpi=160)
    plt.close()


def main() -> None:
    plot_regression_line()
    plot_classification_line()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/generate_enhanced_academic_report.py
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


ROOT = Path(__file__).resolve().parents[1]
REPORTS = ROOT / "reports"
ITER4_DIR = REPORTS / "seven_day_forecast_enhanced_iter4"
ITER20_DIR = REPORTS / "seven_day_forecast_enhanced_iter20"
OUT_DIR = REPORTS / "enhanced_academic_report"
FIG_DIR = OUT_DIR / "figures"
MD_PATH = REPORTS / "combined_weather_academic_report_enhanced.md"


def read_tables(base: Path) -> dict[str, pd.DataFrame]:
    tables = base / "tables"
    return {
        "selected": pd.read_csv(tables / "seven_day_selected_models.csv"),
        "metrics": pd.read_csv(tables / "seven_day_horizon_metrics.csv"),
        "best_params": pd.read_csv(tables / "seven_day_best_hyperparameters.csv"),
        "forecast": pd.read_csv(tables / "seven_day_forecast_predictions.csv"),
        "threshold": pd.read_csv(tables / "seven_day_threshold_tuning.csv"),
        "tuning": pd.read_csv(tables / "seven_day_hyperparameter_tuning.csv"),
    }


def md_table(df: pd.DataFrame, digits: int = 3) -> str:
    data = df.copy()
    for col in data.select_dtypes(include="number").columns:
        data[col] = data[col].round(digits)
    return data.to_markdown(index=False)


def setup() -> None:
    FIG_DIR.mkdir(parents=True, exist_ok=True)


def plot_iter_comparison(iter4: pd.DataFrame, iter20: pd.DataFrame) -> None:
    temp = pd.DataFrame(
        {
            "horizon_day": iter4["horizon_day"],
            "n_iter=4": iter4["test_rmse"],
            "n_iter=20": iter20["test_rmse"],
        }
    )
    plt.figure(figsize=(10, 5))
    plt.plot(temp["horizon_day"], temp["n_iter=4"], marker="o", label="n_iter=4")
    plt.plot(temp["horizon_day"], temp["n_iter=20"], marker="o", label="n_iter=20")
    plt.title("Temperature selected-model test RMSE: n_iter=4 vs n_iter=20")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("Test RMSE (degC)")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "iter4_vs_iter20_temperature_rmse.png", dpi=160)
    plt.close()

    rain = pd.DataFrame(
        {
            "horizon_day": iter4["horizon_day"],
            "n_iter=4": iter4["test_f1"],
            "n_iter=20": iter20["test_f1"],
        }
    )
    plt.figure(figsize=(10, 5))
    plt.plot(rain["horizon_day"], rain["n_iter=4"], marker="o", label="n_iter=4")
    plt.plot(rain["horizon_day"], rain["n_iter=20"], marker="o", label="n_iter=20")
    plt.title("Rain selected-model test F1: n_iter=4 vs n_iter=20")
    plt.xlabel("Forecast horizon day")
    plt.ylabel("Test F1")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "iter4_vs_iter20_rain_f1.png", dpi=160)
    plt.close()


def plot_tuned_model_cv_comparison(tuning4: pd.DataFrame, tuning20: pd.DataFrame) -> None:
    best4 = tuning4[tuning4["is_best"] == True].copy()
    best20 = tuning20[tuning20["is_best"] == True].copy()

    plot_task_cv_comparison(
        best4,
        best20,
        task="temperature_regression",
        metric="mean_cv_rmse",
        ylabel="Best mean CV RMSE",
        filename="per_model_tuning_temperature_cv_rmse_iter4_vs_iter20.png",
        lower_is_better=True,
    )
    plot_task_cv_comparison(
        best4,
        best20,
        task="rain_classification",
        metric="mean_cv_f1",
        ylabel="Best mean CV F1",
        filename="per_model_tuning_rain_cv_f1_iter4_vs_iter20.png",
        lower_is_better=False,
    )


def plot_task_cv_comparison(
    best4: pd.DataFrame,
    best20: pd.DataFrame,
    task: str,
    metric: str,
    ylabel: str,
    filename: str,
    lower_is_better: bool,
) -> None:
    combined = pd.concat([best4, best20], ignore_index=True)
    combined = combined[(combined["task"] == task) & (combined[metric].notna())]
    models = sorted(combined["model"].unique())
    if not models:
        return
    fig, axes = plt.subplots(len(models), 1, figsize=(11, 4 * len(models)), sharex=True)
    if len(models) == 1:
        axes = [axes]

    for ax, model in zip(axes, models):
        for label, data in [("n_iter=4", best4), ("n_iter=20", best20)]:
            part = data[(data["task"] == task) & (data["model"] == model)].sort_values("horizon_day")
            if not part.empty and part[metric].notna().any():
                ax.plot(part["horizon_day"], part[metric], marker="o", label=label)
        direction = "lower is better" if lower_is_better else "higher is better"
        ax.set_title(f"{model}: best cross-validation score by horizon ({direction})")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.25)
        ax.legend()

    axes[-1].set_xlabel("Forecast horizon day")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=160)
    plt.close()


def build_selected_best_params(best_params: pd.DataFrame) -> pd.DataFrame:
    selected = best_params[best_params["selected_model"].notna()].copy()
    cols = [
        "horizon_day",
        "task",
        "model",
        "mean_cv_rmse",
        "mean_cv_f1",
        "selected_metric",
        "selected_metric_value",
        "best_params",
    ]
    return selected[cols]


def write_report(t4: dict[str, pd.DataFrame], t20: dict[str, pd.DataFrame]) -> None:
    selected4 = t4["selected"]
    selected20 = t20["selected"]
    best20_selected = build_selected_best_params(t20["best_params"])
    best4_selected = build_selected_best_params(t4["best_params"])

    comparison = selected4[
        [
            "horizon_day",
            "best_temperature_model",
            "test_rmse",
            "best_rain_model",
            "test_f1",
        ]
    ].rename(
        columns={
            "best_temperature_model": "iter4_temp_model",
            "test_rmse": "iter4_temp_rmse",
            "best_rain_model": "iter4_rain_model",
            "test_f1": "iter4_rain_f1",
        }
    )
    comparison = comparison.merge(
        selected20[
            [
                "horizon_day",
                "best_temperature_model",
                "test_rmse",
                "best_rain_model",
                "test_f1",
            ]
        ].rename(
            columns={
                "best_temperature_model": "iter20_temp_model",
                "test_rmse": "iter20_temp_rmse",
                "best_rain_model": "iter20_rain_model",
                "test_f1": "iter20_rain_f1",
            }
        ),
        on="horizon_day",
    )
    comparison["rmse_delta_iter20_minus_iter4"] = comparison["iter20_temp_rmse"] - comparison["iter4_temp_rmse"]
    comparison["f1_delta_iter20_minus_iter4"] = comparison["iter20_rain_f1"] - comparison["iter4_rain_f1"]

    summary = pd.DataFrame(
        [
            {
                "metric": "Mean selected temperature test RMSE",
                "n_iter=4": selected4["test_rmse"].mean(),
                "n_iter=20": selected20["test_rmse"].mean(),
                "interpretation": "Lower is better",
            },
            {
                "metric": "Mean selected rain test F1",
                "n_iter=4": selected4["test_f1"].mean(),
                "n_iter=20": selected20["test_f1"].mean(),
                "interpretation": "Higher is better",
            },
        ]
    )

    forecast20 = t20["forecast"][
        [
            "horizon_day",
            "forecast_time",
            "temperature_model",
            "predicted_temperature",
            "rain_model",
            "predicted_rain_probability",
            "rain_threshold",
            "predicted_rain_label",
            "predicted_weather_level",
        ]
    ]

    content = f"""---
title: "Báo cáo học thuật nâng cấp: Dự báo thời tiết Hà Nội lúc 12:00 bằng Machine Learning"
author: "ML Final Project"
date: "2026-06-09"
lang: vi
---

# Tóm tắt

Báo cáo này là phiên bản học thuật nâng cấp của hệ thống dự báo thời tiết Hà Nội lúc 12:00. So với bản báo cáo trước, phiên bản này bổ sung bốn nội dung quan trọng: tham số tối ưu được chọn bởi `RandomizedSearchCV`, biểu đồ hyperparameter tuning riêng cho từng model, so sánh trực tiếp giữa mức tìm kiếm `n_iter=4` và `n_iter=20`, và biểu đồ model comparison dạng line để thể hiện xu hướng theo horizon.

Hệ thống giải quyết hai nhiệm vụ học máy: hồi quy nhiệt độ và phân loại mưa. Với dự báo 7 ngày, bài toán được mô hình hóa theo hướng direct multi-horizon: mỗi horizon từ 1 đến 7 có một mô hình nhiệt độ và một mô hình mưa riêng. Dữ liệu được chia theo thời gian: training giai đoạn 2020-2023, validation năm 2024 và test năm 2025. Nguyên tắc đánh giá là chỉ dùng test set sau khi đã hoàn tất chọn mô hình, chọn cấu hình và chọn threshold.

Kết quả cho thấy tăng số cấu hình tìm kiếm từ `n_iter=4` lên `n_iter=20` cải thiện hiệu năng trung bình của pipeline 7 ngày. RMSE test trung bình của nhiệt độ giảm từ {selected4['test_rmse'].mean():.3f} xuống {selected20['test_rmse'].mean():.3f}. F1 test trung bình của phân loại mưa tăng từ {selected4['test_f1'].mean():.3f} lên {selected20['test_f1'].mean():.3f}. Mức cải thiện không đồng đều ở mọi horizon, nhưng có ý nghĩa về mặt thực nghiệm vì phiên bản `n_iter=20` khám phá không gian tham số sâu hơn và cho kết quả tốt hơn trên trung bình toàn bộ 7 ngày.

# 1. Mục tiêu nghiên cứu

Mục tiêu của dự án là xây dựng một pipeline học máy có khả năng dự báo thời tiết Hà Nội lúc 12:00 trong tương lai. Đầu ra chính gồm nhiệt độ dự báo, nhãn mưa/không mưa và mức thời tiết suy ra từ nhiệt độ. Bài toán nhiệt độ là hồi quy, trong khi bài toán mưa là phân loại nhị phân với nhãn được định nghĩa từ điều kiện `precipitation > 0`.

Trong bối cảnh dự báo 7 ngày, horizon 1 tương ứng ngày kế tiếp, horizon 2 tương ứng hai ngày sau, và tiếp tục đến horizon 7. Cách tiếp cận này không dự báo đệ quy từng ngày rồi dùng kết quả ngày trước làm input cho ngày sau. Thay vào đó, mỗi horizon được huấn luyện trực tiếp từ cùng tập đặc trưng lịch sử sang target tương lai tương ứng. Thiết kế này giúp tránh tích lũy lỗi theo chuỗi, nhưng cần huấn luyện nhiều mô hình hơn.

# 2. Dữ liệu và tiền xử lý

Dữ liệu được thu thập từ Open-Meteo cho khu vực Hà Nội trong giai đoạn 2020-2025. Tập dữ liệu ban đầu có 52.608 bản ghi theo giờ. Sau khi chỉ giữ các quan sát lúc 12:00, dữ liệu còn 2.192 bản ghi. Các biến sử dụng gồm nhiệt độ, độ ẩm, lượng mưa, mây che phủ, áp suất mực nước biển, tốc độ gió và bức xạ sóng ngắn.

Quy trình tiền xử lý gồm loại bỏ metadata, chuẩn hóa tên cột, chuyển đổi thời gian sang datetime, sắp xếp theo thời gian, loại bỏ timestamp trùng, giữ các bản ghi 12:00, loại bỏ cột không dùng hoặc có khả năng trùng thông tin, và điền missing numeric bằng median nếu có. Đặc trưng rolling được tính sau khi `shift(1)` để bảo đảm mô hình chỉ sử dụng thông tin quá khứ, không dùng dữ liệu của ngày cần dự báo.

# 3. Đặc trưng và target

Nhóm đặc trưng gồm đặc trưng thời gian, đặc trưng trễ và đặc trưng rolling. Đặc trưng thời gian như `month`, `day_of_year`, `season` giúp mô hình nắm bắt mùa vụ. Đặc trưng lag 1, 3 và 7 ngày giúp mô hình học tính liên tục ngắn hạn của khí tượng. Đặc trưng rolling mô tả xu hướng gần đây của nhiệt độ, độ ẩm và lượng mưa.

Với horizon `h`, target được định nghĩa:

```text
horizon_temperature = temperature.shift(-h)
horizon_rain = precipitation.shift(-h) > 0
```

Thiết kế này làm rõ input/output của mô hình: input là trạng thái khí tượng lịch sử tại 12:00 với đặc trưng đã tạo; output là nhiệt độ và nhãn mưa tại 12:00 sau `h` ngày.

# 4. Thiết kế thực nghiệm

Dữ liệu được chia theo thời gian:

| Tập dữ liệu | Giai đoạn | Vai trò |
|---|---|---|
| Training | 2020-2023 | Huấn luyện trọng số và tuning tham số qua TimeSeriesSplit |
| Validation | 2024 | Chọn model cuối và chọn threshold mưa |
| Test | 2025 | Đánh giá cuối cùng trên dữ liệu chưa dùng để chọn |

Việc tuning trên training set và chọn model trên validation set là cần thiết để tránh overfitting vào test set. Nếu chọn cấu hình trực tiếp trên test set, kết quả sẽ không còn phản ánh năng lực tổng quát hóa của mô hình.

# 5. Hyperparameter tuning

Hyperparameter tuning được thực hiện bằng `RandomizedSearchCV` với `TimeSeriesSplit(n_splits=5)`. `TimeSeriesSplit` giữ đúng thứ tự thời gian: mỗi fold huấn luyện trên một đoạn quá khứ và kiểm tra trên đoạn tương lai ngay sau đó. Đây là lựa chọn phù hợp hơn KFold ngẫu nhiên trong bài toán chuỗi thời gian.

Ở nhóm hồi quy, cấu hình tốt nhất được chọn theo negative RMSE trong cross-validation. Khi báo cáo, giá trị này được đổi dấu thành mean CV RMSE để dễ diễn giải: thấp hơn là tốt hơn. Ở nhóm phân loại, cấu hình tốt nhất được chọn theo F1 trong cross-validation: cao hơn là tốt hơn.

Linear Regression không có hyperparameter tuning trong pipeline này, vì mô hình được fit trực tiếp bằng ordinary least squares sau chuẩn hóa. KNN tune `n_neighbors` và `weights`. XGBoost tune `n_estimators`, `max_depth` và `learning_rate`. Logistic Regression tune `C`. SVM tune `C` và `kernel`.

## 5.1. Tham số tốt nhất của phiên bản n_iter=20

{md_table(best20_selected)}

Các tham số trên là cấu hình tốt nhất trong phạm vi search space của từng model tại từng horizon. Tuy nhiên, một model có best CV score tốt trong training cross-validation chưa chắc là model cuối cùng. Sau tuning, các model đã được cấu hình sẽ được đánh giá trên validation set; model cuối cùng theo horizon được chọn bằng validation RMSE đối với nhiệt độ và validation F1 đối với mưa.

## 5.2. Tham số tốt nhất của phiên bản n_iter=4

{md_table(best4_selected)}

Bảng này là mốc so sánh để thấy khi chỉ thử 4 cấu hình, không gian tham số được khảo sát nông hơn. Với XGBoost, phiên bản `n_iter=4` thường chọn các cấu hình khác và ở nhiều horizon cho RMSE test cao hơn phiên bản `n_iter=20`.

## 5.3. Biểu đồ tuning riêng cho từng model

Các biểu đồ dưới đây mô tả quá trình chọn cấu hình của từng model trong phiên bản `n_iter=20`. Mỗi điểm trên đường biểu diễn một cấu hình được `RandomizedSearchCV` thử. Điểm được đánh dấu đỏ trong file gốc thể hiện cấu hình có rank tốt nhất. Với hồi quy, trục tung là CV RMSE nên thấp hơn là tốt hơn. Với phân loại, trục tung là CV F1 nên cao hơn là tốt hơn.

![KNN tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_temperature_regression_KNNRegressor.png)

![XGBoost Regressor tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_temperature_regression_XGBoostRegressor.png)

![Logistic Regression tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_rain_classification_LogisticRegression.png)

![SVM tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_rain_classification_SVM.png)

![XGBoost Classifier tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_rain_classification_XGBoostClassifier.png)

# 6. So sánh n_iter=4 và n_iter=20

{md_table(summary)}

So sánh theo từng horizon:

{md_table(comparison)}

![So sánh RMSE nhiệt độ giữa n_iter=4 và n_iter=20](enhanced_academic_report/figures/iter4_vs_iter20_temperature_rmse.png)

Biểu đồ cho thấy `n_iter=20` không thắng tuyệt đối ở mọi horizon. Horizon 1 và 2 có RMSE test cao hơn nhẹ so với `n_iter=4`. Tuy nhiên, từ horizon 3 đến horizon 7, `n_iter=20` cải thiện rõ hơn, làm RMSE trung bình toàn bộ 7 ngày thấp hơn. Điều này cho thấy tìm kiếm sâu hơn đặc biệt có ích ở các horizon xa, nơi quan hệ giữa input hiện tại và nhiệt độ tương lai phức tạp hơn.

![So sánh F1 mưa giữa n_iter=4 và n_iter=20](enhanced_academic_report/figures/iter4_vs_iter20_rain_f1.png)

Đối với mưa, cải thiện tập trung ở horizon 1 và horizon 6. Các horizon khác gần như không đổi vì model và threshold được chọn tương tự nhau. Điều này phản ánh rằng tăng số cấu hình tìm kiếm giúp một phần, nhưng bản chất bài toán mưa vẫn khó do tính thưa, tính cục bộ và biến động nhanh của mưa.

![So sánh tuning CV RMSE theo từng model nhiệt độ](enhanced_academic_report/figures/per_model_tuning_temperature_cv_rmse_iter4_vs_iter20.png)

![So sánh tuning CV F1 theo từng model mưa](enhanced_academic_report/figures/per_model_tuning_rain_cv_f1_iter4_vs_iter20.png)

Hai biểu đồ trên giải thích sự khác biệt giữa việc thử 4 cấu hình và thử 20 cấu hình. Với các model có search space rời rạc nhỏ như Logistic Regression và SVM, tăng `n_iter` có thể nhanh chóng chạm giới hạn số cấu hình khả dụng. Với XGBoost, search space lớn hơn nhiều; do đó `n_iter=20` có cơ hội tìm được cấu hình cây tốt hơn so với `n_iter=4`.

# 7. Model comparison dạng line

Theo yêu cầu báo cáo, các biểu đồ model comparison đã được chuyển sang dạng line để thể hiện rõ xu hướng hiệu năng theo forecast horizon.

![Temperature model comparison line chart](seven_day_forecast_enhanced_iter20/figures/seven_day_temperature_model_comparison.png)

Ở bài toán nhiệt độ, line chart cho thấy Linear Regression mạnh ở horizon ngắn nhưng giảm lợi thế khi horizon dài hơn. XGBoostRegressor có xu hướng ổn định hơn trong bài toán 7 ngày sau khi được tuning sâu hơn, nên được chọn ở toàn bộ horizon trong phiên bản `n_iter=20`.

![Rain model comparison line chart](seven_day_forecast_enhanced_iter20/figures/seven_day_rain_model_comparison.png)

Ở bài toán mưa, line chart cho thấy F1 dao động mạnh hơn nhiệt độ. Không có một mô hình thống trị tuyệt đối mọi horizon. Logistic Regression được chọn ở nhiều horizon vì đạt validation F1 tốt và ổn định, trong khi XGBoostClassifier tốt hơn ở horizon 1 và 6, SVM tốt ở horizon 7.

# 8. Threshold tuning cho phân loại mưa

Sau khi mỗi mô hình phân loại sinh xác suất mưa, pipeline không dùng cố định threshold 0,5. Thay vào đó, threshold được thử từ 0,10 đến 0,90 với bước 0,05 trên validation set. Threshold tốt nhất là threshold làm F1 validation cao nhất.

![Threshold tuning by horizon](seven_day_forecast_enhanced_iter20/figures/rain_threshold_tuning_by_horizon.png)

Biểu đồ threshold tuning cho thấy mỗi horizon có một ngưỡng quyết định khác nhau. Ngưỡng thấp thường làm tăng recall vì mô hình dễ dự báo mưa hơn, nhưng có thể làm giảm precision vì tăng false positive. Ngưỡng cao thường cải thiện precision nhưng có nguy cơ bỏ sót ngày mưa. Do đó, threshold tối ưu là điểm cân bằng theo F1, không nhất thiết bằng 0,5.

# 9. Kết quả dự báo 7 ngày của phiên bản n_iter=20

{md_table(forecast20)}

![Dự báo nhiệt độ 7 ngày](seven_day_forecast_enhanced_iter20/figures/next_7_days_temperature_forecast.png)

![Dự báo mưa 7 ngày](seven_day_forecast_enhanced_iter20/figures/next_7_days_rain_forecast.png)

Kết quả dự báo được sinh từ bản ghi 12:00 mới nhất trong dữ liệu, tức 2025-12-30 12:00. Bảy ngày dự báo cụ thể là từ 2025-12-31 đến 2026-01-06. Phiên bản `n_iter=20` dự báo toàn bộ 7 ngày là không mưa tại thời điểm 12:00, với xác suất mưa đều thấp hơn threshold đã chọn. Nhiệt độ dự báo chuyển từ mức normal ở ngày đầu sang cool ở các ngày sau.

# 10. Đánh giá actual-predicted trên test set

![Actual vs predicted temperature on full test set](seven_day_forecast_enhanced_iter20/figures/test_actual_vs_predicted_temperature_best_models_by_horizon.png)

Biểu đồ nhiệt độ actual-predicted trải dài toàn bộ test set năm 2025 cho thấy mô hình bám được xu hướng mùa vụ và dao động chính của nhiệt độ. Sai lệch tăng ở các horizon xa, phù hợp với nguyên lý dự báo: càng xa hiện tại, độ bất định càng lớn.

![Actual vs predicted rain on full test set](seven_day_forecast_enhanced_iter20/figures/test_actual_vs_predicted_rain_best_models_by_horizon.png)

Biểu đồ mưa actual-predicted cho thấy mô hình phân loại mưa còn sai đáng kể. Điều này không mâu thuẫn với F1 ở mức trung bình, vì mưa là sự kiện thưa và khó tách. Để cải thiện đáng kể bài toán mưa, cần thêm dữ liệu dự báo khí tượng tương lai, dữ liệu nhiều thời điểm trong ngày hoặc dữ liệu không gian từ khu vực lân cận.

# 11. Thảo luận học thuật

Kết quả thực nghiệm cho thấy vai trò của hyperparameter tuning phụ thuộc vào độ phức tạp của model và kích thước search space. Với Linear Regression, không có tham số chính cần tìm kiếm trong pipeline hiện tại, nên hiệu năng phụ thuộc chủ yếu vào chất lượng đặc trưng và tính tuyến tính của quan hệ dữ liệu. Với KNN, tuning ảnh hưởng đến độ mượt của dự báo thông qua số láng giềng và trọng số khoảng cách. Với XGBoost, tuning có ảnh hưởng lớn hơn vì số cây, độ sâu và learning rate quyết định năng lực mô hình hóa phi tuyến cũng như rủi ro overfitting.

Việc tăng `n_iter` từ 4 lên 20 làm rõ hơn quá trình lựa chọn cấu hình. Khi chỉ thử 4 cấu hình, kết quả có thể phụ thuộc mạnh vào các mẫu ngẫu nhiên ban đầu. Khi thử 20 cấu hình, xác suất tìm được cấu hình phù hợp hơn tăng lên, nhất là với XGBoost. Tuy nhiên, tăng `n_iter` không bảo đảm mọi horizon đều tốt hơn vì validation/test có thể khác phân phối, và mô hình được chọn theo validation không được tối ưu trực tiếp trên test.

Về mặt phương pháp luận, test set không được dùng để chọn model, tham số hoặc threshold. Điều này làm kết quả có tính học thuật hơn vì test set giữ vai trò đánh giá cuối cùng. Một số trường hợp như Logistic Regression có thể tốt hơn SVM trên test ở bài toán ngày kế tiếp, nhưng nếu không thắng trên validation thì không được chọn làm mô hình cuối theo quy trình nghiêm ngặt.

# 12. Kết luận

Phiên bản báo cáo nâng cấp đã bổ sung đầy đủ thông tin về tham số tốt nhất, biểu đồ tuning riêng cho từng model, biểu đồ so sánh `n_iter=4` và `n_iter=20`, và biểu đồ model comparison dạng line. Kết quả tổng hợp cho thấy `n_iter=20` là phiên bản nên dùng làm báo cáo chính vì cải thiện RMSE trung bình của nhiệt độ và F1 trung bình của mưa trên toàn bộ 7 horizon.

Tuy nhiên, mức cải thiện là vừa phải, không mang tính đột phá. Điều này cho thấy giới hạn chính của bài toán không chỉ nằm ở số lượng cấu hình tuning, mà còn nằm ở loại dữ liệu đầu vào. Để tăng đáng kể độ chính xác, đặc biệt với mưa, cần bổ sung các biến khí tượng tương lai từ mô hình dự báo số trị, dữ liệu nhiều thời điểm trong ngày, dữ liệu radar/mây hoặc dữ liệu không gian từ các khu vực xung quanh Hà Nội.
"""
    MD_PATH.write_text(content, encoding="utf-8")


def main() -> None:
    setup()
    t4 = read_tables(ITER4_DIR)
    t20 = read_tables(ITER20_DIR)
    plot_iter_comparison(t4["selected"], t20["selected"])
    plot_tuned_model_cv_comparison(t4["tuning"], t20["tuning"])
    write_report(t4, t20)


if __name__ == "__main__":
    main()


## 4. Chạy pipeline dự báo ngày kế tiếp

Pipeline này tạo các kết quả chính trong `reports/figures`, `reports/tables` và `reports/final_results.md`. Kết quả tương ứng với phần dự báo ngày kế tiếp trong final academic report.


In [ ]:
!python main.py


## 5. Sinh biểu đồ model comparison dạng line cho bài toán ngày kế tiếp


In [ ]:
!python scripts/generate_next_day_line_comparison_charts.py


## 6. Chạy pipeline dự báo 7 ngày với `n_iter=4`

Đây là mốc so sánh ban đầu, dùng số lần lấy mẫu cấu hình thấp hơn trong `RandomizedSearchCV`.


In [ ]:
import os
os.environ['SEVEN_DAY_OUTPUT_DIR'] = 'seven_day_forecast_enhanced_iter4'
os.environ['SEVEN_DAY_SEARCH_ITER'] = '4'
!python seven_day_pipeline.py


## 7. Chạy pipeline dự báo 7 ngày với `n_iter=20`

Đây là phiên bản tuned chính trong báo cáo cuối. Cell này có thể chạy lâu hơn do thử nhiều cấu hình hơn.


In [ ]:
import os
os.environ['SEVEN_DAY_OUTPUT_DIR'] = 'seven_day_forecast_enhanced_iter20'
os.environ['SEVEN_DAY_SEARCH_ITER'] = '20'
!python seven_day_pipeline.py


## 8. Tạo báo cáo enhanced và biểu đồ so sánh `n_iter=4` vs `n_iter=20`


In [ ]:
!python scripts/generate_enhanced_academic_report.py


## 9. Tạo final academic report tổng hợp

Cell này ghi lại bản báo cáo final academic report tổng hợp dưới dạng Markdown. Báo cáo sử dụng các bảng số liệu và hình ảnh được sinh ra từ các pipeline ở trên.


In [ ]:
from pathlib import Path
Path('reports').mkdir(parents=True, exist_ok=True)
final_report = '---\ntitle: "Báo cáo học thuật tổng hợp cuối: Dự báo thời tiết Hà Nội lúc 12:00 bằng Machine Learning"\nauthor: "ML Final Project"\ndate: "2026-06-09"\nlang: vi\n---\n\n# Tóm tắt\n\nBáo cáo này tổng hợp và hệ thống hóa kết quả từ ba báo cáo đã xây dựng trước đó: `hanoi_noon_weather_academic_report.md`, `combined_weather_academic_report.md` và `combined_weather_academic_report_enhanced.md`. Mục tiêu là tạo ra một bản báo cáo học thuật thống nhất, trình bày đầy đủ từ bài toán dự báo ngày kế tiếp đến bài toán dự báo 7 ngày, đồng thời bổ sung phân tích hyperparameter tuning, threshold tuning và so sánh giữa hai mức tìm kiếm cấu hình `n_iter=4` và `n_iter=20`.\n\nNghiên cứu sử dụng dữ liệu Open-Meteo cho khu vực Hà Nội trong giai đoạn 2020-2025. Tập dữ liệu ban đầu có 52.608 bản ghi theo giờ; sau tiền xử lý chỉ giữ quan sát lúc 12:00, dữ liệu còn 2.192 bản ghi. Hệ thống giải quyết hai nhiệm vụ học máy: dự báo nhiệt độ dưới dạng hồi quy và dự báo mưa dưới dạng phân loại nhị phân. Với bài toán ngày kế tiếp, Linear Regression đạt kết quả tốt nhất cho nhiệt độ với RMSE test 2,319°C và R2 0,808. Với bài toán mưa ngày kế tiếp, SVM được chọn theo F1 trên validation set và đạt F1 test 0,673.\n\nĐối với dự báo 7 ngày, pipeline được mở rộng theo hướng direct multi-horizon: mỗi horizon từ 1 đến 7 có một mô hình nhiệt độ và một mô hình mưa riêng. Phiên bản tuned tăng số cấu hình `RandomizedSearchCV` từ 4 lên 20, làm RMSE test trung bình của nhiệt độ giảm từ 3,290 xuống 3,159 và F1 test trung bình của mưa tăng từ 0,625 lên 0,632. Kết quả cho thấy việc tuning sâu hơn có cải thiện, đặc biệt ở các horizon xa, nhưng không giải quyết hoàn toàn khó khăn của bài toán mưa do mưa là hiện tượng thưa, cục bộ và biến động mạnh.\n\n# 1. Giới thiệu\n\nDự báo thời tiết là một bài toán có giá trị thực tiễn cao trong sinh hoạt, giao thông, giáo dục, lao động ngoài trời và quản lý đô thị. Trong nghiên cứu này, hệ thống tập trung vào thời điểm 12:00 trưa tại Hà Nội. Việc cố định thời điểm dự báo làm cho bài toán có mục tiêu rõ ràng hơn, tránh trộn lẫn biến động theo giờ và giúp các mô hình học máy tập trung vào quan hệ giữa trạng thái khí tượng lịch sử và thời tiết tương lai tại cùng một mốc thời gian trong ngày.\n\nHai bài toán học máy được đặt ra:\n\n- Dự báo nhiệt độ lúc 12:00, là bài toán hồi quy.\n- Dự báo có mưa hay không lúc 12:00, là bài toán phân loại nhị phân.\n\nBên cạnh hai đầu ra chính, hệ thống suy ra mức thời tiết từ nhiệt độ dự báo:\n\n```text\ntemperature < 22        -> cool\n22 <= temperature <= 30 -> normal\ntemperature > 30        -> hot\n```\n\nMức thời tiết không được huấn luyện bằng mô hình riêng vì đây là biến suy diễn trực tiếp từ nhiệt độ. Cách thiết kế này làm giảm độ phức tạp không cần thiết và tránh tạo thêm một nguồn sai số độc lập.\n\n# 2. Dữ liệu nghiên cứu\n\n## 2.1. Nguồn dữ liệu\n\nDữ liệu được thu thập từ Open-Meteo cho khu vực Hà Nội, gồm các biến khí tượng như nhiệt độ, độ ẩm tương đối, lượng mưa, mây che phủ, áp suất mực nước biển, tốc độ gió ở độ cao 10m và bức xạ sóng ngắn. Khoảng thời gian dữ liệu là từ ngày 2020-01-01 đến ngày 2025-12-31.\n\n| Thuộc tính | Giá trị |\n|---|---:|\n| Số bản ghi theo giờ ban đầu | 52.608 |\n| Số bản ghi lúc 12:00 sau tiền xử lý | 2.192 |\n| Số mẫu supervised cho bài toán ngày kế tiếp | 2.191 |\n| Khoảng thời gian dữ liệu | 2020-01-01 đến 2025-12-31 |\n| Tỷ lệ mưa ngày kế tiếp | 0,390 |\n| Bản ghi mới nhất dùng cho suy luận 7 ngày | 2025-12-30 12:00:00 |\n\n## 2.2. Thống kê mô tả\n\n| Biến | Mean | Std | Min | Median | Max |\n|---|---:|---:|---:|---:|---:|\n| temperature | 26,906 | 5,497 | 8,900 | 27,800 | 38,000 |\n| humidity | 67,167 | 12,306 | 22,000 | 68,000 | 96,000 |\n| precipitation | 0,255 | 0,957 | 0,000 | 0,000 | 13,800 |\n| cloud_cover | 77,693 | 33,656 | 0,000 | 99,000 | 100,000 |\n| pressure_msl | 1011,833 | 7,343 | 990,500 | 1011,300 | 1032,300 |\n| wind_speed_10m | 9,766 | 5,027 | 0,000 | 9,300 | 38,000 |\n| shortwave_radiation | 547,390 | 235,568 | 59,000 | 568,500 | 949,000 |\n\nNhiệt độ trung bình lúc 12:00 là 26,906°C với độ lệch chuẩn 5,497°C. Điều này cho thấy biến thiên mùa vụ rõ rệt: các giai đoạn lạnh có thể xuống dưới 10°C, trong khi mùa nóng có thể đạt 38°C vào buổi trưa. Lượng mưa có median bằng 0 mm, nghĩa là ít nhất một nửa số ngày không có mưa tại thời điểm 12:00. Độ lệch chuẩn lượng mưa lớn hơn mean, phản ánh phân phối lệch phải: phần lớn quan sát có mưa bằng 0 hoặc rất nhỏ, trong khi một số ngày có mưa lớn. Đặc điểm này làm cho bài toán phân loại mưa khó hơn bài toán hồi quy nhiệt độ.\n\nCloud cover có median 99%, cho thấy nhiều quan sát buổi trưa có mây che phủ cao. Shortwave radiation dao động từ 59 đến 949, do đó biến này được đưa vào thí nghiệm A/B ở pipeline ngày kế tiếp để kiểm tra tác động đến hiệu quả dự báo.\n\n# 3. Tiền xử lý và tạo đặc trưng\n\nQuy trình tiền xử lý gồm các bước:\n\n1. Phát hiện và loại bỏ các dòng metadata của Open-Meteo.\n2. Chuẩn hóa tên cột.\n3. Chuyển cột thời gian sang kiểu datetime.\n4. Sắp xếp dữ liệu theo thời gian.\n5. Loại bỏ timestamp trùng lặp.\n6. Chỉ giữ các quan sát lúc 12:00.\n7. Loại bỏ các cột không dùng hoặc có nguy cơ trùng thông tin, gồm `surface_pressure`, `wind_direction_10m` và `rain`.\n8. Điền missing numeric bằng median nếu có.\n9. Loại bỏ các dòng không có target sau khi tạo nhãn tương lai.\n\nCột `rain` bị loại bỏ vì có thể trùng lặp thông tin với `precipitation`; trong khi đó, nhãn mưa được định nghĩa lại từ lượng mưa tương lai theo điều kiện `precipitation > 0`. Quyết định này giúp giảm rủi ro rò rỉ dữ liệu và làm cho nhãn phân loại nhất quán.\n\nCác nhóm đặc trưng chính gồm:\n\n- Đặc trưng thời gian: `month`, `day_of_year`, `season`.\n- Đặc trưng trễ: lag 1, 3 và 7 ngày cho nhiệt độ, độ ẩm và lượng mưa.\n- Đặc trưng rolling: trung bình trượt nhiệt độ, trung bình trượt độ ẩm và tổng lượng mưa trên cửa sổ 3 và 7 ngày.\n\nCác đặc trưng rolling được tính bằng `shift(1)` trước khi áp dụng `rolling()`. Đây là điểm quan trọng về phương pháp luận vì nó bảo đảm mô hình chỉ sử dụng thông tin quá khứ, không dùng dữ liệu của chính ngày cần dự báo.\n\n# 4. Trực quan hóa dữ liệu\n\n## 4.1. Xu hướng và phân phối nhiệt độ\n\n![Xu hướng nhiệt độ lúc 12:00](figures/temperature_trend.png)\n\nBiểu đồ xu hướng nhiệt độ cho thấy chu kỳ mùa vụ rõ rệt qua các năm. Nhiệt độ tăng trong các tháng nóng và giảm trong các giai đoạn lạnh. Đây là cơ sở để đưa các đặc trưng thời gian như `month`, `season` và `day_of_year` vào mô hình.\n\n![Phân phối nhiệt độ](figures/temperature_histogram.png)\n\nPhân phối nhiệt độ cho thấy phần lớn quan sát tập trung quanh vùng nhiệt độ trung bình đến nóng, phù hợp với đặc điểm khí hậu Hà Nội.\n\n## 4.2. Correlation heatmap\n\n![Correlation heatmap](figures/correlation_heatmap.png)\n\nCorrelation heatmap mô tả quan hệ tuyến tính giữa các biến. Các đặc trưng nhiệt độ hiện tại, nhiệt độ lag và rolling temperature có tương quan cao. Điều này phù hợp với bản chất vật lý của nhiệt độ: trạng thái nhiệt hôm nay thường chịu ảnh hưởng mạnh từ những ngày gần trước đó. Tuy nhiên, tương quan cao cũng đặt ra vấn đề đa cộng tuyến khi diễn giải hệ số Linear Regression. Do đó, hệ số tuyến tính và feature importance cần được hiểu như tín hiệu mô hình, không nên diễn giải trực tiếp như quan hệ nhân quả.\n\n## 4.3. PCA\n\n![PCA visualization](figures/pca_visualization.png)\n\nPCA giảm không gian đặc trưng nhiều chiều xuống hai thành phần chính PC1 và PC2. Các giá trị trên trục PCA như -4, -2, 0, 2, 4 là tọa độ trong không gian mới sau phép biến đổi tuyến tính, không còn mang đơn vị khí tượng gốc như °C, mm hay hPa. Nếu các điểm mưa và không mưa chồng lấn nhiều trong không gian PCA, điều đó cho thấy bài toán phân loại mưa khó tách bằng một ranh giới đơn giản trong hai chiều biểu diễn chính.\n\n# 5. Thiết kế thực nghiệm\n\n## 5.1. Chia train, validation và test\n\nDữ liệu được chia theo thứ tự thời gian thay vì chia ngẫu nhiên:\n\n| Tập dữ liệu | Giai đoạn | Vai trò |\n|---|---|---|\n| Training | 2020-2023 | Huấn luyện trọng số và tuning tham số bằng TimeSeriesSplit |\n| Validation | 2024 | Chọn model cuối và chọn threshold mưa |\n| Test | 2025 | Đánh giá cuối cùng trên dữ liệu chưa dùng để chọn |\n\nChia theo thời gian là bắt buộc trong bài toán dự báo vì mô hình chỉ được học từ quá khứ để dự đoán tương lai. Nếu chia ngẫu nhiên, dữ liệu tương lai có thể xuất hiện trong training set, làm kết quả đánh giá bị lạc quan giả tạo.\n\n## 5.2. Baseline\n\nHai baseline được sử dụng làm mốc so sánh:\n\n- Temperature baseline: dự báo nhiệt độ ngày mai bằng nhiệt độ hôm nay. Đây là persistence baseline, phù hợp với tính liên tục của nhiệt độ.\n- Rain baseline: luôn dự báo lớp phổ biến nhất trong training set, tương đương `DummyClassifier(strategy="most_frequent")`.\n\nBaseline không phải là mô hình học máy phức tạp. Vai trò của baseline là kiểm tra xem các mô hình học máy có thực sự tạo thêm giá trị so với chiến lược dự báo đơn giản hay không.\n\n## 5.3. Mô hình\n\nNhóm hồi quy nhiệt độ gồm:\n\n- Linear Regression.\n- KNN Regressor.\n- XGBoost Regressor.\n\nNhóm phân loại mưa gồm:\n\n- Logistic Regression với `class_weight=\'balanced\'`.\n- SVM với `class_weight=\'balanced\'`.\n- XGBoost Classifier với `scale_pos_weight` tự động theo tỷ lệ lớp.\n\nCác mô hình nhạy với thang đo như Linear Regression, Logistic Regression, SVM và KNN được kết hợp với `StandardScaler`. XGBoost là mô hình cây nên không yêu cầu chuẩn hóa đặc trưng.\n\n# 6. Hyperparameter tuning và threshold tuning\n\nHyperparameter tuning được thực hiện bằng `RandomizedSearchCV` kết hợp `TimeSeriesSplit(n_splits=5)` trên training set. `TimeSeriesSplit` giữ đúng thứ tự thời gian: mỗi fold huấn luyện trên một đoạn quá khứ và kiểm tra trên đoạn tương lai ngay sau đó. Điều này phù hợp với bản chất chuỗi thời gian hơn KFold ngẫu nhiên.\n\nỞ bài toán hồi quy, cấu hình tốt nhất được chọn theo negative RMSE trong cross-validation. Khi báo cáo, giá trị này được đổi dấu thành mean CV RMSE, trong đó thấp hơn là tốt hơn. Ở bài toán phân loại, cấu hình tốt nhất được chọn theo F1 trong cross-validation, trong đó cao hơn là tốt hơn.\n\nLinear Regression không có hyperparameter tuning trong pipeline hiện tại vì mô hình được fit bằng ordinary least squares sau chuẩn hóa. KNN tune `n_neighbors` và `weights`. XGBoost tune `n_estimators`, `max_depth` và `learning_rate`. Logistic Regression tune `C`. SVM tune `C` và `kernel`.\n\nSau khi mô hình phân loại sinh xác suất mưa, pipeline không sử dụng cố định threshold 0,5. Threshold được thử từ 0,10 đến 0,90 với bước 0,05 trên validation set; threshold tốt nhất là threshold tối đa hóa F1 validation. Như vậy, nhóm classification có hai lớp tuning: tuning tham số mô hình trên training cross-validation và tuning ngưỡng quyết định trên validation set.\n\n# 7. Kết quả dự báo ngày kế tiếp\n\n## 7.1. Hồi quy nhiệt độ\n\nKết quả validation:\n\n| Mô hình | MAE | RMSE | R2 |\n|---|---:|---:|---:|\n| Linear Regression | 1,811 | 2,375 | 0,814 |\n| KNN Regressor | 2,214 | 2,761 | 0,749 |\n| XGBoost Regressor | 1,893 | 2,439 | 0,804 |\n\nKết quả test:\n\n| Mô hình | MAE | RMSE | R2 |\n|---|---:|---:|---:|\n| Linear Regression | 1,674 | 2,319 | 0,808 |\n| KNN Regressor | 2,148 | 2,821 | 0,716 |\n| XGBoost Regressor | 1,830 | 2,432 | 0,789 |\n\nLinear Regression được chọn vì có RMSE thấp nhất trên validation set. Trên test set, mô hình này tiếp tục đạt RMSE thấp nhất và R2 cao nhất. Kết quả này cho thấy dự báo nhiệt độ ngày kế tiếp lúc 12:00 có cấu trúc tương đối tuyến tính, trong đó nhiệt độ hiện tại, đặc trưng mùa vụ, lag và rolling đã chứa phần lớn tín hiệu dự báo.\n\n![So sánh mô hình hồi quy ngày kế tiếp dạng line](figures/regression_model_comparison_line.png)\n\n![Actual vs predicted temperature trên test set](figures/test_actual_vs_predicted_temperature.png)\n\n![Temperature residuals trên test set](figures/test_temperature_residuals.png)\n\nĐối với hồi quy, không nên đánh giá đúng/sai tuyệt đối như phân loại. Vì vậy, báo cáo dùng thêm các ngưỡng sai số:\n\n| Tập dữ liệu | MAE | RMSE | Trong ±1°C | Trong ±2°C | Trong ±3°C |\n|---|---:|---:|---:|---:|---:|\n| Validation | 1,811 | 2,375 | 0,388 | 0,653 | 0,806 |\n| Test | 1,674 | 2,319 | 0,420 | 0,692 | 0,849 |\n\nTrên test set, khoảng 69,2% dự báo nằm trong sai số ±2°C và 84,9% nằm trong ±3°C. Đây là kết quả hợp lý với một hệ thống sử dụng dữ liệu lịch sử tại một điểm không gian.\n\n## 7.2. Trọng số w và hệ số chặn b của Linear Regression\n\nĐối với Linear Regression, quá trình huấn luyện không tìm kiếm hyperparameter như `n_neighbors`, `max_depth` hay `learning_rate`. Thay vào đó, mô hình ước lượng trực tiếp bộ trọng số `w` và hệ số chặn `b` bằng cách cực tiểu hóa hàm mất mát bình phương sai số trên training set. Công thức dự báo có dạng:\n\n```text\ny_hat = w1*x1 + w2*x2 + ... + wn*xn + b\n```\n\nTrong đó `x1 ... xn` là các đặc trưng đầu vào sau chuẩn hóa, `w1 ... wn` là các hệ số tuyến tính học được, và `b` là hệ số chặn của mô hình. Vì pipeline sử dụng `StandardScaler` trước Linear Regression, các hệ số `w` được diễn giải trong không gian đặc trưng đã chuẩn hóa. Do đó, độ lớn của hệ số cho biết mức đóng góp tương đối của đặc trưng sau chuẩn hóa, không phải mức thay đổi trực tiếp theo đơn vị gốc như °C, mm hay hPa.\n\nCác hệ số có độ lớn cao nhất trong mô hình nhiệt độ ngày kế tiếp gồm:\n\n| Đặc trưng | Hệ số w | Hướng tác động |\n|---|---:|---|\n| temperature | 4,485 | positive |\n| day_of_year | -1,007 | negative |\n| temp_lag1 | -0,881 | negative |\n| month | 0,851 | positive |\n| temp_lag7 | 0,540 | positive |\n| humidity | 0,391 | positive |\n| temp_roll_mean_3 | 0,285 | positive |\n| pressure_msl | -0,267 | negative |\n| temp_lag3 | 0,214 | positive |\n| season | 0,208 | positive |\n\nHệ số dương nghĩa là khi đặc trưng đó tăng trong không gian đã chuẩn hóa, dự báo nhiệt độ có xu hướng tăng nếu các đặc trưng khác giữ nguyên. Hệ số âm nghĩa là đặc trưng đó có quan hệ ngược chiều với nhiệt độ dự báo trong mô hình tuyến tính. Ví dụ, hệ số của `temperature` lớn và dương cho thấy nhiệt độ hiện tại là tín hiệu mạnh nhất cho nhiệt độ ngày kế tiếp. Các đặc trưng mùa vụ như `day_of_year`, `month` và `season` cũng có hệ số đáng kể, phản ánh chu kỳ khí hậu theo năm.\n\nĐiểm cần nhấn mạnh là `w` và `b` không phải là “tham số tuned tốt nhất” theo nghĩa hyperparameter tuning. Chúng là tham số mô hình được học trực tiếp từ dữ liệu training. Bảng đầy đủ các hệ số được lưu trong `reports/tables/linear_regression_coefficients.csv`, còn biểu đồ hệ số được trình bày ở phần diễn giải mô hình.\n\n## 7.3. Phân loại mưa\n\nKết quả validation:\n\n| Mô hình | Accuracy | Precision | Recall | F1 | ROC AUC |\n|---|---:|---:|---:|---:|---:|\n| Logistic Regression | 0,653 | 0,532 | 0,712 | 0,609 | 0,742 |\n| SVM | 0,680 | 0,560 | 0,741 | 0,638 | 0,758 |\n| XGBoost Classifier | 0,686 | 0,574 | 0,669 | 0,618 | 0,751 |\n\nKết quả test:\n\n| Mô hình | Accuracy | Precision | Recall | F1 | ROC AUC |\n|---|---:|---:|---:|---:|---:|\n| Logistic Regression | 0,720 | 0,612 | 0,783 | 0,687 | 0,785 |\n| SVM | 0,717 | 0,616 | 0,741 | 0,673 | 0,774 |\n| XGBoost Classifier | 0,698 | 0,611 | 0,636 | 0,623 | 0,767 |\n\nSVM được chọn vì có F1 cao nhất trên validation set. Trên test set, Logistic Regression có F1 cao hơn SVM, nhưng lựa chọn mô hình sau khi nhìn test set sẽ vi phạm quy trình đánh giá. Do đó, SVM vẫn là mô hình được chọn hợp lệ theo nguyên tắc validation-first.\n\n![So sánh mô hình phân loại ngày kế tiếp dạng line](figures/classification_model_comparison_line.png)\n\n![Confusion matrix trên test set](figures/test_confusion_matrix_rain.png)\n\nConfusion matrix cho biết mô hình dự báo đúng/sai ở từng lớp. False negative nghĩa là thực tế có mưa nhưng mô hình dự báo không mưa; false positive nghĩa là thực tế không mưa nhưng mô hình dự báo có mưa. Tùy ứng dụng, false negative có thể nghiêm trọng hơn vì người dùng không chuẩn bị cho mưa.\n\n## 7.4. Thí nghiệm A/B với shortwave radiation\n\n| Thí nghiệm | Dùng shortwave | Best regression | Validation RMSE | Test RMSE | Best rain | Validation F1 | Test F1 |\n|---|---:|---|---:|---:|---|---:|---:|\n| A_without_shortwave_radiation | False | Linear Regression | 2,374 | 2,317 | SVM | 0,652 | 0,660 |\n| B_with_shortwave_radiation | True | Linear Regression | 2,375 | 2,319 | SVM | 0,638 | 0,673 |\n\nĐối với nhiệt độ, thêm `shortwave_radiation` không cải thiện rõ rệt; RMSE gần như không đổi. Đối với mưa, không dùng shortwave tốt hơn trên validation nhưng dùng shortwave tốt hơn trên test. Điều này cho thấy biến shortwave có thể chứa tín hiệu khí tượng, nhưng tác động của nó chưa ổn định giữa các năm.\n\n![So sánh A/B](figures/experiment_ab_comparison.png)\n\n# 8. Kết quả dự báo 7 ngày\n\n## 8.1. Thiết kế direct multi-horizon\n\nVới mỗi horizon `h = 1..7`, target được định nghĩa:\n\n```text\nhorizon_temperature = temperature.shift(-h)\nhorizon_rain = precipitation.shift(-h) > 0\n```\n\nĐây là direct multi-horizon forecasting. Mỗi horizon có một cặp mô hình riêng cho nhiệt độ và mưa. Cách này tránh lỗi tích lũy của phương pháp recursive, nhưng cần huấn luyện nhiều mô hình hơn.\n\n## 8.2. Phiên bản n_iter=4\n\n| Horizon | Mô hình nhiệt độ | RMSE test | Mô hình mưa | F1 test | Threshold mưa |\n|---:|---|---:|---|---:|---:|\n| 1 | LinearRegression | 2,319 | XGBoostClassifier | 0,652 | 0,20 |\n| 2 | LinearRegression | 3,010 | LogisticRegression | 0,653 | 0,40 |\n| 3 | XGBoostRegressor | 3,436 | LogisticRegression | 0,638 | 0,45 |\n| 4 | XGBoostRegressor | 3,682 | LogisticRegression | 0,617 | 0,40 |\n| 5 | XGBoostRegressor | 3,747 | LogisticRegression | 0,587 | 0,50 |\n| 6 | XGBoostRegressor | 3,422 | LogisticRegression | 0,597 | 0,40 |\n| 7 | XGBoostRegressor | 3,417 | SVM | 0,634 | 0,40 |\n\nSai số nhiệt độ tăng khi horizon dài hơn. Đây là xu hướng phù hợp với lý thuyết dự báo: dự báo càng xa hiện tại thì độ bất định càng lớn. Với mưa, F1 dao động quanh 0,587-0,653, cho thấy bài toán mưa 7 ngày khó hơn do tính thưa và biến động nhanh của sự kiện mưa.\n\n## 8.3. Phiên bản n_iter=20\n\n| Horizon | Mô hình nhiệt độ | RMSE validation | RMSE test | Mô hình mưa | F1 validation | F1 test | Threshold |\n|---:|---|---:|---:|---|---:|---:|---:|\n| 1 | XGBoostRegressor | 2,375 | 2,358 | XGBoostClassifier | 0,672 | 0,678 | 0,35 |\n| 2 | XGBoostRegressor | 3,100 | 3,061 | LogisticRegression | 0,668 | 0,653 | 0,40 |\n| 3 | XGBoostRegressor | 3,337 | 3,283 | LogisticRegression | 0,647 | 0,638 | 0,45 |\n| 4 | XGBoostRegressor | 3,459 | 3,415 | LogisticRegression | 0,641 | 0,617 | 0,40 |\n| 5 | XGBoostRegressor | 3,436 | 3,512 | LogisticRegression | 0,628 | 0,587 | 0,50 |\n| 6 | XGBoostRegressor | 3,458 | 3,262 | XGBoostClassifier | 0,615 | 0,620 | 0,45 |\n| 7 | XGBoostRegressor | 3,516 | 3,220 | SVM | 0,620 | 0,634 | 0,40 |\n\nTrong phiên bản `n_iter=20`, XGBoostRegressor được chọn cho toàn bộ 7 horizon nhiệt độ. Điều này cho thấy khi dự báo xa hơn một ngày, quan hệ giữa input hiện tại/quá khứ và nhiệt độ tương lai không còn tuyến tính đơn giản như bài toán ngày kế tiếp. XGBoost có khả năng mô hình hóa tương tác phi tuyến giữa mùa, độ ẩm, mây, bức xạ, lag và rolling, nên phù hợp hơn với bài toán đa horizon.\n\n![Temperature model comparison line chart](seven_day_forecast_enhanced_iter20/figures/seven_day_temperature_model_comparison.png)\n\n![Rain model comparison line chart](seven_day_forecast_enhanced_iter20/figures/seven_day_rain_model_comparison.png)\n\n## 8.4. So sánh n_iter=4 và n_iter=20\n\n| Chỉ số | n_iter=4 | n_iter=20 | Diễn giải |\n|---|---:|---:|---|\n| Mean selected temperature test RMSE | 3,290 | 3,159 | Thấp hơn là tốt hơn |\n| Mean selected rain test F1 | 0,625 | 0,632 | Cao hơn là tốt hơn |\n\n| Horizon | RMSE n_iter=4 | RMSE n_iter=20 | Delta RMSE | F1 n_iter=4 | F1 n_iter=20 | Delta F1 |\n|---:|---:|---:|---:|---:|---:|---:|\n| 1 | 2,319 | 2,358 | +0,039 | 0,652 | 0,678 | +0,026 |\n| 2 | 3,010 | 3,061 | +0,051 | 0,653 | 0,653 | 0,000 |\n| 3 | 3,436 | 3,283 | -0,153 | 0,638 | 0,638 | 0,000 |\n| 4 | 3,682 | 3,415 | -0,267 | 0,617 | 0,617 | 0,000 |\n| 5 | 3,747 | 3,512 | -0,235 | 0,587 | 0,587 | 0,000 |\n| 6 | 3,422 | 3,262 | -0,160 | 0,597 | 0,620 | +0,023 |\n| 7 | 3,417 | 3,220 | -0,198 | 0,634 | 0,634 | 0,000 |\n\n![So sánh RMSE nhiệt độ giữa n_iter=4 và n_iter=20](enhanced_academic_report/figures/iter4_vs_iter20_temperature_rmse.png)\n\n![So sánh F1 mưa giữa n_iter=4 và n_iter=20](enhanced_academic_report/figures/iter4_vs_iter20_rain_f1.png)\n\nPhiên bản `n_iter=20` không tốt hơn ở mọi horizon. Ở horizon 1 và 2, RMSE nhiệt độ tăng nhẹ. Tuy nhiên, từ horizon 3 đến 7, RMSE giảm rõ hơn, làm RMSE trung bình toàn bộ 7 ngày thấp hơn. Với mưa, cải thiện tập trung ở horizon 1 và 6; các horizon khác gần như không đổi. Như vậy, tuning sâu hơn có ích nhưng không phải yếu tố duy nhất quyết định chất lượng dự báo.\n\n# 9. Tham số tốt nhất và diễn giải tuning\n\n## 9.1. Tham số tốt nhất của phiên bản n_iter=20\n\n| Horizon | Task | Model | CV metric | Best params |\n|---:|---|---|---:|---|\n| 1 | temperature | XGBoostRegressor | RMSE 2,896 | learning_rate=0,01597; max_depth=4; n_estimators=443 |\n| 1 | rain | XGBoostClassifier | F1 0,638 | learning_rate=0,01597; max_depth=4; n_estimators=443 |\n| 2 | temperature | XGBoostRegressor | RMSE 3,546 | learning_rate=0,01597; max_depth=4; n_estimators=443 |\n| 2 | rain | LogisticRegression | F1 0,579 | C=0,01 |\n| 3 | temperature | XGBoostRegressor | RMSE 3,988 | learning_rate=0,01597; max_depth=4; n_estimators=443 |\n| 3 | rain | LogisticRegression | F1 0,562 | C=0,01 |\n| 4 | temperature | XGBoostRegressor | RMSE 4,138 | learning_rate=0,01597; max_depth=4; n_estimators=443 |\n| 4 | rain | LogisticRegression | F1 0,546 | C=0,01 |\n| 5 | temperature | XGBoostRegressor | RMSE 4,004 | learning_rate=0,12177; max_depth=4; n_estimators=152 |\n| 5 | rain | LogisticRegression | F1 0,562 | C=0,01 |\n| 6 | temperature | XGBoostRegressor | RMSE 3,952 | learning_rate=0,01597; max_depth=4; n_estimators=443 |\n| 6 | rain | XGBoostClassifier | F1 0,507 | learning_rate=0,01597; max_depth=4; n_estimators=443 |\n| 7 | temperature | XGBoostRegressor | RMSE 3,837 | learning_rate=0,01597; max_depth=4; n_estimators=443 |\n| 7 | rain | SVM | F1 0,549 | C=0,1; kernel=linear |\n\nBảng trên trình bày cấu hình tốt nhất trong search space của model được chọn tại từng horizon. Cần phân biệt hai giai đoạn: `RandomizedSearchCV` chọn cấu hình tốt nhất cho từng model trên training cross-validation; sau đó validation set chọn model cuối cùng giữa các candidate model đã được tuning.\n\n## 9.2. Biểu đồ hyperparameter tuning từng model\n\n![KNN tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_temperature_regression_KNNRegressor.png)\n\n![XGBoost Regressor tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_temperature_regression_XGBoostRegressor.png)\n\n![Logistic Regression tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_rain_classification_LogisticRegression.png)\n\n![SVM tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_rain_classification_SVM.png)\n\n![XGBoost Classifier tuning](seven_day_forecast_enhanced_iter20/figures/hyperparameter_tuning_rain_classification_XGBoostClassifier.png)\n\nCác biểu đồ này cho thấy mỗi điểm tương ứng một cấu hình được thử bởi `RandomizedSearchCV`. Với hồi quy, trục tung là CV RMSE nên càng thấp càng tốt. Với phân loại, trục tung là CV F1 nên càng cao càng tốt. Việc so sánh đường tuning giúp thấy rõ model nào nhạy với cấu hình tham số và model nào ổn định hơn.\n\n![So sánh tuning CV RMSE theo từng model nhiệt độ](enhanced_academic_report/figures/per_model_tuning_temperature_cv_rmse_iter4_vs_iter20.png)\n\n![So sánh tuning CV F1 theo từng model mưa](enhanced_academic_report/figures/per_model_tuning_rain_cv_f1_iter4_vs_iter20.png)\n\nVới các model có search space nhỏ như Logistic Regression và SVM, tăng `n_iter` có thể nhanh chóng chạm giới hạn số cấu hình khả dụng. Với XGBoost, search space lớn hơn nhiều; do đó `n_iter=20` có cơ hội tìm được cấu hình cây tốt hơn so với `n_iter=4`.\n\n# 10. Kết quả suy luận 7 ngày tương lai\n\nKết quả dự báo được sinh từ bản ghi 12:00 mới nhất trong dữ liệu, tức 2025-12-30 12:00. Bảy ngày dự báo cụ thể là từ 2025-12-31 đến 2026-01-06.\n\n| Ngày dự báo | Mô hình nhiệt độ | Nhiệt độ dự báo | Mô hình mưa | Xác suất mưa | Threshold | Nhãn mưa | Mức thời tiết |\n|---|---|---:|---|---:|---:|---|---|\n| 2025-12-31 | XGBoostRegressor | 22,951 | XGBoostClassifier | 0,314 | 0,35 | no_rain | normal |\n| 2026-01-01 | XGBoostRegressor | 20,880 | LogisticRegression | 0,229 | 0,40 | no_rain | cool |\n| 2026-01-02 | XGBoostRegressor | 20,150 | LogisticRegression | 0,241 | 0,45 | no_rain | cool |\n| 2026-01-03 | XGBoostRegressor | 18,374 | LogisticRegression | 0,268 | 0,40 | no_rain | cool |\n| 2026-01-04 | XGBoostRegressor | 16,657 | LogisticRegression | 0,261 | 0,50 | no_rain | cool |\n| 2026-01-05 | XGBoostRegressor | 17,840 | XGBoostClassifier | 0,127 | 0,45 | no_rain | cool |\n| 2026-01-06 | XGBoostRegressor | 18,283 | SVM | 0,285 | 0,40 | no_rain | cool |\n\n![Dự báo nhiệt độ 7 ngày](seven_day_forecast_enhanced_iter20/figures/next_7_days_temperature_forecast.png)\n\n![Dự báo mưa 7 ngày](seven_day_forecast_enhanced_iter20/figures/next_7_days_rain_forecast.png)\n\nMô hình dự báo toàn bộ 7 ngày là không mưa tại thời điểm 12:00 vì xác suất mưa đều thấp hơn threshold đã chọn. Nhiệt độ dự báo chuyển từ mức normal ở ngày đầu sang cool ở các ngày tiếp theo.\n\n# 11. Đánh giá actual-predicted trên test set\n\n![Actual vs predicted temperature on full test set](seven_day_forecast_enhanced_iter20/figures/test_actual_vs_predicted_temperature_best_models_by_horizon.png)\n\nBiểu đồ nhiệt độ actual-predicted trên toàn bộ test set năm 2025 cho thấy mô hình bám được xu hướng mùa vụ và dao động chính của nhiệt độ. Sai lệch tăng ở các horizon xa, phù hợp với nguyên lý dự báo: càng xa hiện tại, độ bất định càng lớn.\n\n![Actual vs predicted rain on full test set](seven_day_forecast_enhanced_iter20/figures/test_actual_vs_predicted_rain_best_models_by_horizon.png)\n\nBiểu đồ mưa actual-predicted cho thấy mô hình phân loại mưa còn sai đáng kể. Điều này phù hợp với F1 ở mức trung bình, vì mưa là sự kiện thưa và khó tách. Để cải thiện đáng kể bài toán mưa, cần thêm dữ liệu dự báo khí tượng tương lai, dữ liệu nhiều thời điểm trong ngày hoặc dữ liệu không gian từ khu vực lân cận.\n\n# 12. Feature importance và diễn giải mô hình\n\nFeature importance cần được hiểu theo từng loại mô hình. Với Linear Regression và Logistic Regression, hệ số sau chuẩn hóa cho biết chiều và độ mạnh tương đối của ảnh hưởng tuyến tính. Với XGBoost, importance phản ánh mức độ đặc trưng được dùng trong các cây để giảm lỗi. Với KNN và SVM, permutation importance đo mức suy giảm hiệu năng khi tráo một đặc trưng.\n\n![XGBoost Regressor feature importance](figures/xgboost_regressor_feature_importance.png)\n\n![Linear Regression coefficients](figures/linear_regression_coefficients.png)\n\n![SVM permutation importance](figures/svm_permutation_importance.png)\n\nFeature importance không phản ánh model nào là tốt nhất. Chất lượng mô hình phải được đánh giá bằng các metric trên validation/test như RMSE, MAE, R2, F1 và ROC AUC. Feature importance chỉ trả lời câu hỏi mô hình đang dựa nhiều vào đặc trưng nào trong quá trình ra quyết định.\n\n# 13. Diễn giải các chỉ số đánh giá\n\nMAE là sai số tuyệt đối trung bình. Với nhiệt độ, MAE 1,674 nghĩa là trung bình dự báo lệch khoảng 1,674°C so với thực tế. MAE dễ hiểu nhưng không phạt mạnh các lỗi lớn.\n\nRMSE là căn bậc hai của sai số bình phương trung bình. RMSE phạt các lỗi lớn mạnh hơn MAE; vì vậy RMSE thấp hơn nghĩa là mô hình ít mắc các lỗi dự báo lệch xa.\n\nR2 đo tỷ lệ phương sai của biến mục tiêu được mô hình giải thích. R2 gần 1 là tốt; R2 gần 0 nghĩa là mô hình không tốt hơn nhiều so với việc dự báo bằng giá trị trung bình. Trong dự báo 7 ngày, R2 giảm khi horizon dài hơn là hiện tượng hợp lý.\n\nWithin ±1°C, ±2°C và ±3°C cho biết tỷ lệ dự báo nằm trong một khoảng sai số chấp nhận được. Chỉ số này hữu ích hơn cách nói đúng/sai tuyệt đối trong hồi quy vì nhiệt độ là biến liên tục.\n\nAccuracy là tỷ lệ dự báo đúng tổng thể trong phân loại mưa. Precision cho biết trong các lần mô hình dự báo mưa, bao nhiêu lần thực sự có mưa. Recall cho biết trong các ngày thực sự có mưa, mô hình phát hiện được bao nhiêu. F1 là trung bình điều hòa giữa precision và recall, phù hợp khi dữ liệu mất cân bằng. ROC AUC đo khả năng xếp hạng xác suất giữa lớp mưa và không mưa.\n\n# 14. Thảo luận học thuật\n\nKết quả thực nghiệm cho thấy bài toán nhiệt độ dễ dự báo hơn bài toán mưa. Nguyên nhân chính là nhiệt độ có tính liên tục theo thời gian và chịu ảnh hưởng mạnh của mùa vụ. Ngược lại, mưa là hiện tượng cục bộ, rời rạc và chịu ảnh hưởng của nhiều yếu tố khí quyển mà dữ liệu một điểm lúc 12:00 không thể mô tả đầy đủ.\n\nTrong bài toán ngày kế tiếp, Linear Regression đạt kết quả tốt nhất cho nhiệt độ. Điều này không có nghĩa Linear Regression luôn mạnh hơn XGBoost, mà phản ánh rằng với horizon ngắn và tập dữ liệu hiện có, quan hệ tuyến tính đã đủ hiệu quả. Trong bài toán 7 ngày, XGBoostRegressor chiếm ưu thế vì dự báo xa hơn yêu cầu mô hình hóa quan hệ phi tuyến và tương tác đặc trưng phức tạp hơn.\n\nViệc tăng `n_iter` từ 4 lên 20 làm rõ vai trò của hyperparameter tuning. Khi chỉ thử 4 cấu hình, kết quả có thể phụ thuộc mạnh vào các mẫu ngẫu nhiên ban đầu. Khi thử 20 cấu hình, xác suất tìm được cấu hình phù hợp hơn tăng lên, đặc biệt với XGBoost. Tuy nhiên, tăng `n_iter` không bảo đảm mọi horizon đều tốt hơn vì validation/test có thể khác phân phối, và mô hình được chọn theo validation không được tối ưu trực tiếp trên test.\n\nĐối với mưa, tuning threshold là cần thiết. Nếu dùng threshold mặc định 0,5, mô hình có thể quá bảo thủ và dự báo nhiều ngày không mưa. Threshold tối ưu theo validation set giúp cân bằng precision và recall theo F1. Tuy nhiên, threshold tuning chỉ điều chỉnh quyết định đầu ra, không thay thế được việc cần thêm đặc trưng khí tượng mạnh hơn.\n\n# 15. Hạn chế\n\nThứ nhất, dữ liệu chỉ sử dụng một điểm không gian là Hà Nội và chỉ lấy thời điểm 12:00. Điều này làm bài toán rõ ràng nhưng mất thông tin về diễn biến theo giờ và cấu trúc không gian của thời tiết.\n\nThứ hai, pipeline dự báo 7 ngày dùng direct multi-horizon từ bản ghi hiện tại, không tự sinh toàn bộ chuỗi biến khí hậu tương lai như độ ẩm, áp suất, gió, mây hay bức xạ. Vì vậy, mô hình đang học quan hệ từ trạng thái hiện tại/quá khứ sang target tương lai, chứ không mô phỏng khí quyển đầy đủ.\n\nThứ ba, test set chỉ là năm 2025. Mặc dù đánh giá này đúng quy trình, kết quả vẫn có thể chịu ảnh hưởng bởi đặc điểm riêng của năm đó. Nhiều năm test hơn sẽ giúp kết luận ổn định hơn.\n\nThứ tư, bài toán mưa được chuyển thành nhị phân `precipitation > 0`. Trong thực tế, mưa rất nhỏ và mưa đáng kể có ý nghĩa sử dụng khác nhau. Một hướng cải thiện là dự báo nhiều mức mưa hoặc dự báo lượng mưa liên tục rồi suy ra nhãn.\n\n# 16. Kết luận và hướng phát triển\n\nDự án đã xây dựng thành công pipeline học máy cho dự báo thời tiết Hà Nội lúc 12:00, bao gồm dự báo ngày kế tiếp và dự báo 7 ngày. Hệ thống thực hiện đầy đủ các bước của một quy trình học máy học thuật: tiền xử lý dữ liệu, phân tích thống kê, trực quan hóa, tạo đặc trưng không rò rỉ dữ liệu, chia train/validation/test theo thời gian, tuning tham số bằng `TimeSeriesSplit`, lựa chọn mô hình bằng validation set, đánh giá cuối trên test set và sinh báo cáo kết quả.\n\nVới bài toán ngày kế tiếp, Linear Regression là mô hình tốt nhất cho nhiệt độ với RMSE test 2,319°C và R2 0,808. Với bài toán mưa ngày kế tiếp, SVM được chọn theo validation F1 và đạt F1 test 0,673. Với bài toán dự báo 7 ngày, phiên bản `n_iter=20` là phiên bản thuyết phục hơn vì cải thiện RMSE trung bình của nhiệt độ từ 3,290 xuống 3,159 và F1 trung bình của mưa từ 0,625 lên 0,632.\n\nTrong tương lai, hệ thống có thể được cải thiện bằng cách bổ sung dữ liệu nhiều thời điểm trong ngày, dữ liệu không gian từ các khu vực lân cận, biến khí tượng dự báo từ mô hình số trị, hoặc dữ liệu ảnh mây/radar. Ngoài ra, có thể thử các mô hình chuỗi thời gian như LSTM, Temporal Convolutional Network hoặc Transformer, nhưng cần đánh giá cẩn thận để tránh overfitting trên tập dữ liệu còn tương đối nhỏ.\n'
Path('reports/final_synthesized_academic_report.md').write_text(final_report, encoding='utf-8')
print('Wrote reports/final_synthesized_academic_report.md')


## 10. Xuất DOCX từ final academic report

Cell này dùng pandoc để xuất file Word. Nếu chỉ cần notebook/bảng/hình, có thể bỏ qua cell này.


In [ ]:
!pandoc reports/final_synthesized_academic_report.md --resource-path=reports -o reports/final_synthesized_academic_report.docx
!ls -lh reports/final_synthesized_academic_report.md reports/final_synthesized_academic_report.docx


## 11. Xem nhanh các kết quả chính


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

print('Next-day temperature regression results')
display(pd.read_csv('reports/tables/regression_results.csv'))

print('Next-day rain classification results')
display(pd.read_csv('reports/tables/rain_classification_results.csv'))

print('7-day selected models: n_iter=20')
display(pd.read_csv('reports/seven_day_forecast_enhanced_iter20/tables/seven_day_selected_models.csv'))

print('7-day forecast output: n_iter=20')
display(pd.read_csv('reports/seven_day_forecast_enhanced_iter20/tables/seven_day_forecast_predictions.csv'))


## 12. Hiển thị một số biểu đồ quan trọng


In [ ]:
from IPython.display import Image, display

figures = [
    'reports/figures/regression_model_comparison_line.png',
    'reports/figures/classification_model_comparison_line.png',
    'reports/seven_day_forecast_enhanced_iter20/figures/seven_day_temperature_model_comparison.png',
    'reports/seven_day_forecast_enhanced_iter20/figures/seven_day_rain_model_comparison.png',
    'reports/enhanced_academic_report/figures/iter4_vs_iter20_temperature_rmse.png',
    'reports/enhanced_academic_report/figures/iter4_vs_iter20_rain_f1.png',
    'reports/seven_day_forecast_enhanced_iter20/figures/rain_threshold_tuning_by_horizon.png',
]
for fig in figures:
    print(fig)
    display(Image(filename=fig))


## 13. Tải các artifact từ Colab

Sau khi chạy xong, bạn có thể tải file báo cáo `.docx`, `.md`, các bảng CSV hoặc toàn bộ thư mục `reports`.


In [ ]:
# Nén toàn bộ kết quả để tải về từ Colab.
!zip -qr weather_ml_reports.zip reports
!ls -lh weather_ml_reports.zip

try:
    from google.colab import files
    files.download('weather_ml_reports.zip')
except Exception as exc:
    print('Nếu không chạy trong Colab, hãy lấy file weather_ml_reports.zip trong thư mục hiện tại.')
    print(exc)
